In [1]:
import heapq
from bisect import bisect_right
from collections import defaultdict

# 1) Stack

**The gist.** A stack hands things back in the order you need to come back to them — the most recent unfinished thing is always on top.

> **Think of it like…** a pile of plates. Opening brackets are plates you put down; a closing bracket must take the top plate, and it had better be its partner.

| Family | Stack holds | Pop when |
|---|---|---|
| **Matching / evaluation** | open brackets, partial results, saved context | the matching closer or operator arrives |
| **Monotonic stack** | indices with increasing (or decreasing) values | a new value breaks the order — each popped item just found its *next greater / smaller* |

```python
stack = []                          # indices, values decreasing
for i, x in enumerate(nums):
    while stack and nums[stack[-1]] < x:
        j = stack.pop()             # x is the next greater element of nums[j]
    stack.append(i)
```

**Spot it:** brackets / "valid" / nested `k[...]` / paths with `..` · calculators · "next greater / warmer / smaller", "span", "how many days until" · histogram rectangles · "remove k digits for the smallest result".

**Classic bugs:** popping an empty stack · forgetting leftovers after the loop · pushing values when you need distances (push indices) · `//` floors negatives — use `int(a / b)` to truncate.

## 1.1 Matching and evaluation

Ask: *what's the most recent unfinished thing, and what finishes it?* That tells you what to push and when to pop. When nesting shows up (`(`, `[`), push the whole current context and start fresh.

### 1.1.1 [20. Valid Parentheses](https://leetcode.com/problems/valid-parentheses/) — ★★★ · Easy

**Problem.** Is a string of `()[]{}` correctly matched and nested?  
**Example.** `"()[]{}"` → `True`; `"(]"` → `False`  
**Spot it.** Most recent open bracket must close first → stack.

**Ask first**
- Other characters besides brackets? → If yes, skip them.

> 💡 **Idea.** Push openers. A closer must match the top of the stack; at the end the stack must be empty.

**Brute force:** repeatedly delete `()`, `[]`, `{}` pairs until nothing changes, O(n²).

**Steps**
1. Push every opening bracket.
2. On a closing bracket, the stack must be non-empty and its top must be the partner.
3. Pop the partner and continue.
4. Valid iff the stack is empty at the end.

**Cost:** Time O(n) (each character pushed/popped at most once) · Space O(n) (all openers in the worst case)

In [ ]:
PAIRS = {")": "(", "]": "[", "}": "{"}


def is_valid(s):
    """True if brackets are balanced and properly nested."""
    stack = []
    for ch in s:
        if ch in PAIRS:
            # closer must match the most recent unclosed opener
            if not stack or stack.pop() != PAIRS[ch]:
                return False
        else:
            stack.append(ch)
    
    # leftover openers were never closed
    return not stack


assert is_valid("()[]{}")
assert not is_valid("(]")
assert is_valid("([])")
assert not is_valid("([)]")
assert not is_valid("(")
print(is_valid("{[()]}"))

True


### 1.1.2 [155. Min Stack](https://leetcode.com/problems/min-stack/) — ★★ · Medium

**Problem.** Stack with `push`, `pop`, `top` and `get_min`, all O(1).  
**Example.** `push(-2)`, `push(0)`, `push(-3)`, `get_min()` → `-3`, `pop()`, `top()` → `0`, `get_min()` → `-2`  
**Spot it.** Minimum must survive pops → store the running minimum alongside each element.

**Ask first**
- Duplicate minimums? → Yes — the pair approach handles them (a separate min-stack must push on `<=`, not `<`).

> 💡 **Idea.** Store each value together with the minimum *at that moment*: push `(x, min(x, current_min))`. Popping brings the old minimum back for free.

**Brute force:** scan the stack for the min on every `get_min`, O(n).

**Steps**
1. Each entry is `(value, minimum of the stack up to here)`.
2. `push`: the new minimum is `min(val, previous minimum)`.
3. `pop` removes the pair, which automatically restores the older minimum.
4. `get_min` reads the top pair.

**Cost:** Time O(1) per operation · Space O(n) (one extra number per entry)

**Likely follow-ups**
- **Max stack instead?** Same trick with `max`. If they also want `pop_max` (LC 716): doubly linked list + sorted map (or two heaps with lazy deletion), O(log n).
- **O(1) extra space (no second stack / pairs)?** Store `x − min` on the stack. If the stored diff is negative, `x` was the new min; on pop, recover the old min as `min − diff`. Watch for overflow in fixed-width ints.
- **Min *queue* instead of stack?** Two min-stacks (in/out) like Queue-from-Stacks, or a monotonic deque.

In [3]:
class MinStack:
    """Stack that reports its minimum in O(1)."""

    def __init__(self):
        """Create an empty stack.

        Each entry is a pair (value, minimum of the stack up to and including
        this entry), so the minimum is always stored next to the top.
        """
        self.items = []

    def push(self, val):
        """Add val on top. O(1).

        The new minimum is the smaller of val and the minimum stored below it
        (or val itself if the stack is empty).
        """
        low = min(val, self.items[-1][1]) if self.items else val
        self.items.append((val, low))

    def pop(self):
        """Remove the top value. O(1).

        Nothing else needs updating: the entry below already holds the older
        minimum, which becomes the current one again.
        """
        self.items.pop()

    def top(self):
        """Return the top value without removing it. O(1)."""
        return self.items[-1][0]

    def get_min(self):
        """Return the smallest value currently in the stack. O(1).

        It is read from the top entry, which stores the minimum of everything
        beneath it as well.
        """
        return self.items[-1][1]


st = MinStack()
for v in [-2, 0, -3]:
    st.push(v)
assert st.get_min() == -3
st.pop()
assert st.top() == 0 and st.get_min() == -2
print(st.get_min())

-2


### 1.1.3 [150. Evaluate Reverse Polish Notation](https://leetcode.com/problems/evaluate-reverse-polish-notation/) — ★★ · Medium

**Problem.** Evaluate a postfix expression of integers and `+ - * /` (division truncates toward 0).  
**Example.** `["2", "1", "+", "3", "*"]` → `9` (that is (2 + 1) × 3)  
**Spot it.** Postfix → operands on a stack, operator pops two.

**Ask first**
- How does division round? → Toward zero → use `int(a / b)`, not `//` (which floors negatives).

> 💡 **Idea.** Numbers go on the stack. An operator pops two (second popped is the left operand), applies, pushes the result.

**Brute force:** repeatedly find the first operator and collapse it with the two tokens before it, O(n²).

**Steps**
1. Push numbers.
2. On an operator, pop `b` (right operand), then `a` (left operand).
3. Push `a op b`, using `int(a / b)` for division.
4. The final stack value is the answer.

**Cost:** Time O(n) · Space O(n)

**Likely follow-ups**
- **Convert infix (`3 + 4 * 2`) to RPN first?** Shunting-yard: output numbers; for operators, pop higher-or-equal precedence ops from an operator stack first; `(` pushes, `)` pops until `(`.
- **Go the other way — print RPN as infix with minimal brackets?** Stack of `(expr_string, precedence)`; wrap an operand in brackets only if its precedence is lower than the operator joining it.
- **Add unary minus or functions like `max(a, b)`?** Give them their own tokens with an arity: pop 1 for unary, pop k for a k-arg function.

In [4]:
def eval_rpn(tokens):
    """Value of a postfix expression."""
    stack = []
    for tok in tokens:
        if tok in "+-*/":
            # right operand is on top
            b, a = stack.pop(), stack.pop()
            if tok == "+":
                stack.append(a + b)
            elif tok == "-":
                stack.append(a - b)
            elif tok == "*":
                stack.append(a * b)
            else:
                # truncate toward zero, unlike //
                stack.append(int(a / b))
        else:
            stack.append(int(tok))
    return stack[0]


assert eval_rpn(["2", "1", "+", "3", "*"]) == 9
assert eval_rpn(["4", "13", "5", "/", "+"]) == 6
tokens = ["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"]
assert eval_rpn(tokens) == 22
print(eval_rpn(tokens))

22


### 1.1.4 [232. Implement Queue using Stacks](https://leetcode.com/problems/implement-queue-using-stacks/) — ★★ · Easy

**Problem.** FIFO queue using only two stacks; amortised O(1) operations.  
**Example.** `push(1)`, `push(2)`, `peek()` → `1`, `pop()` → `1`, `empty()` → `False`  
**Spot it.** Reversing a stack once gives queue order → inbox/outbox.

**Ask first**
- O(1) worst case, or amortised? → Amortised is the target.

> 💡 **Idea.** Push into an `in` stack. To pop, take from an `out` stack; refill `out` from `in` only when `out` is empty. Each item moves at most once → amortised O(1).

**Brute force:** move everything to the other stack on every pop, O(n) per pop.

**Steps**
1. `push` goes onto `inbox`.
2. `pop`/`peek`: if `outbox` is empty, move all of `inbox` into it.
3. Serve from the top of `outbox`.

**Cost:** Time Amortised O(1) · Space O(n)

In [5]:
class MyQueue:
    """Queue built from two stacks."""

    def __init__(self):
        """Create an empty queue.

        inbox collects new items (newest on top). outbox serves items to the
        caller in queue order (oldest on top).
        """
        self.inbox = []
        self.outbox = []

    def push(self, x):
        """Add x to the back of the queue. O(1)."""
        self.inbox.append(x)

    def _shift(self):
        """Make sure outbox holds the front of the queue, if there is one.

        Refills only when outbox is empty: popping everything from inbox into
        outbox reverses the order, so the oldest item ends up on top.
        Each item moves at most once, so pop and peek are O(1) amortized.
        """
        if not self.outbox:
            while self.inbox:
                self.outbox.append(self.inbox.pop())

    def pop(self):
        """Remove and return the item at the front of the queue.

        O(1) amortized. Assumes the queue is not empty.
        """
        self._shift()
        return self.outbox.pop()

    def peek(self):
        """Return the front item without removing it.

        O(1) amortized. Assumes the queue is not empty.
        """
        self._shift()
        return self.outbox[-1]

    def empty(self):
        """Return True if the queue holds no items. O(1)."""
        return not self.inbox and not self.outbox


q = MyQueue()
q.push(1)
q.push(2)
assert q.peek() == 1 and q.pop() == 1 and not q.empty()
print(q.pop(), q.empty())

2 True


### 1.1.5 [71. Simplify Path](https://leetcode.com/problems/simplify-path/) — ★★ · Medium

**Problem.** Canonicalise a Unix path (handle `.`, `..`, repeated slashes).  
**Example.** `"/home//foo/../bar/"` → `"/home/bar"`  
**Spot it.** Directory nesting → stack of path components.

**Ask first**
- Is `...` special? → No, only `.` and `..` are — `...` is a normal name.
- `..` at the root? → Stay at the root.

> 💡 **Idea.** Split on `/`. Skip empty parts and `.`; `..` pops a folder (if any); anything else is a folder name to push. Join with `/`.

**Brute force:** repeatedly regex-replace `/./`, `//`, `/x/../` until stable — fiddly and O(n²).

**Steps**
1. Split on `/`.
2. Skip empty parts and `.`.
3. `..` pops if possible; any other name is pushed.
4. Return `"/" + "/".join(stack)`.

**Cost:** Time O(n) · Space O(n)

**Likely follow-ups**
- **Path can be relative (`cd` from a current dir)?** Start the stack with the current directory's parts, then apply the same rules. (Meta's common variant.)
- **Symlinks exist?** You can't do it with strings alone — resolve each component against the filesystem (a map link → target), restarting from the target, and guard against loops.
- **O(1) extra space?** Two-pointer write over a char array, backing the write pointer up to the previous `/` on `..`.

In [6]:
def simplify_path(path):
    """Canonical absolute path."""
    stack = []
    for part in path.split("/"):
        # go up one level (ignored at the root)
        if part == "..":
            if stack:
                stack.pop()
        # a real directory name ("" and "." change nothing)
        elif part and part != ".":
            stack.append(part)
    return "/" + "/".join(stack)


assert simplify_path("/home/") == "/home"
assert simplify_path("/home//foo/") == "/home/foo"
assert simplify_path("/home/user/Documents/../Pictures") == "/home/user/Pictures"
assert simplify_path("/../") == "/"
assert simplify_path("/.../a/../b/c/../d/./") == "/.../b/d"
print(simplify_path("/a/./b/../../c/"))

/c


### 1.1.6 [227. Basic Calculator II](https://leetcode.com/problems/basic-calculator-ii/) — ★★ · Medium

**Problem.** Evaluate `+ - * /` with non-negative integers and spaces, no parentheses.  
**Example.** `"3+2*2"` → `7`  
**Spot it.** Precedence without parentheses → apply `*`/`/` immediately, defer `+`/`−` on a stack.

**Ask first**
- Parentheses? → None here (that's LC 224 / 772).
- Division? → Truncates toward zero → `int(a / b)`.

> 💡 **Idea.** Remember the last operator. When a number ends: `+` pushes it, `-` pushes `−num`, `*` / `/` combine with the stack top right away. Answer = sum of the stack.

**Brute force:** two passes: first collapse `*` and `/`, then add/subtract — works, just more code.

**Steps**
1. Accumulate a number's digits.
2. When an operator or the end arrives, apply the *previous* operator to that number.
3. `+` pushes `num`, `−` pushes `−num`, `*`/`/` replace the top with `top op num`.
4. Return the sum of the stack.

**Cost:** Time O(n) · Space O(n) (O(1) if you keep only the last term and a running sum)

**Likely follow-ups**
- **O(1) extra space?** Drop the stack: keep `result` and `last_term`. `+`/`-` → `result += last_term; last_term = ±num`. `*`/`/` → `last_term = last_term op num`. Return `result + last_term`.
- **Add parentheses (LC 772)?** On `(` recurse (or push the current state and reset); on `)` return the inner value as a number.
- **Support exponent `^` (right-associative)?** Use shunting-yard with precedence; for right-associative ops pop only *strictly* higher precedence.

In [7]:
def calculate_ii(s):
    """Evaluate an expression with + - * /."""
    # stack of signed terms; their sum is the answer
    stack = []
    num, op = 0, "+"
    for i, ch in enumerate(s):
        if ch.isdigit():
            num = num * 10 + int(ch)
        # number finished: apply the operator that came before it
        if ch in "+-*/" or i == len(s) - 1:
            if op == "+":
                stack.append(num)
            elif op == "-":
                stack.append(-num)
            elif op == "*":
                stack.append(stack.pop() * num)
            else:
                stack.append(int(stack.pop() / num))
            num, op = 0, ch
    return sum(stack)


assert calculate_ii("3+2*2") == 7
assert calculate_ii(" 3/2 ") == 1
assert calculate_ii(" 3+5 / 2 ") == 5
assert calculate_ii("14-3/2") == 13
print(calculate_ii("3+2*2"))

7


### 1.1.7 [224. Basic Calculator](https://leetcode.com/problems/basic-calculator/) — ★★ · Hard

**Problem.** Evaluate `+`, `−`, parentheses and unary minus.  
**Example.** `"(1+(4+5+2)-3)+(6+8)"` → `23`  
**Spot it.** Only `+`/`−` → running sum with a sign; parentheses → push context.

**Ask first**
- Unary minus like `-(2+3)`? → Yes — treat it as `0 − …` (the sign variable does that naturally).

> 💡 **Idea.** Only `+`, `-` and brackets. Keep a running `result` and `sign`. On `(` push `(result, sign)` and reset; on `)` pop and combine: `result = old_result + old_sign * result`.

**Brute force:** `eval` — not allowed.

**Steps**
1. Keep `result`, current `num`, and `sign` (+1/−1).
2. On `+`/`−`, add `sign * num` to `result` and set the new sign.
3. On `(`, push `result` and `sign`, then reset both.
4. On `)`, finish the inner result and combine: `saved_result + saved_sign * inner`.

**Cost:** Time O(n) · Space O(depth) (two numbers per open parenthesis)

**Likely follow-ups**
- **Add `*` and `/` too (LC 772)?** Combine 224's bracket handling with 227's last-term trick, or recurse on each bracket.

In [8]:
def calculate(s):
    """Evaluate an expression with + - and parentheses."""
    stack = []
    result, num, sign = 0, 0, 1
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)
        # close the current number and remember the next sign
        elif ch in "+-":
            result += sign * num
            num, sign = 0, 1 if ch == "+" else -1
        # save the outer context and start a fresh sub-expression
        elif ch == "(":
            stack.extend([result, sign])
            result, sign = 0, 1
        # finish the sub-expression and merge it into the outer one
        elif ch == ")":
            result += sign * num
            num = 0
            saved_sign, saved_result = stack.pop(), stack.pop()
            result = saved_result + saved_sign * result
    return result + sign * num


assert calculate("1 + 1") == 2
assert calculate(" 2-1 + 2 ") == 3
assert calculate("(1+(4+5+2)-3)+(6+8)") == 23
assert calculate("-(2+3)") == -5
print(calculate("(1+(4+5+2)-3)+(6+8)"))

23


### 1.1.8 [394. Decode String](https://leetcode.com/problems/decode-string/) — ★★ · Medium

**Problem.** Decode `k[encoded]` (nested), e.g. `3[a2[c]]` → `accaccacc`.  
**Example.** `"3[a2[c]]"` → `"accaccacc"`  
**Spot it.** Nested brackets → stack of (previous string, repeat count).

**Ask first**
- Multi-digit counts like `12[a]`? → Yes, build the number digit by digit.
- Output size? → Guaranteed ≤ 10⁵. Without that guarantee, `9[9[9[…]]]` can blow up — ask.

> 💡 **Idea.** On `[`, push the current string and the repeat count, then start a fresh string. On `]`, pop and do `prev + count * current`.

**Brute force:** repeatedly find the innermost `k[...]` and expand it, O(output · depth).

**Steps**
1. Digits accumulate into `count`.
2. `[` pushes `(current, count)` and resets both.
3. `]` pops `(prev, k)` and sets `current = prev + current * k`.
4. Letters append to `current`.

**Cost:** Time O(output length) · Space O(output length)

**Likely follow-ups**
- **Output could be gigantic — return just its length, or the k-th char (LC 880)?** Compute lengths without building: length × count per bracket. For the k-th char, walk backwards dividing out the repeats (`k %= size`).
- **Write the encoder (shortest encoding, LC 471)?** Interval DP: for each substring, try splitting it and try `k[pattern]` if it's a repeat (`(s+s).find(s, 1)` finds the period).
- **Recursive version?** `decode(i)` returns `(string, next_index)`; on a digit read k, recurse after `[`, repeat the result.

In [9]:
def decode_string(s):
    """Expand k[...] encodings."""
    stack = []
    current, count = "", 0
    for ch in s:
        # multi-digit repeat counts
        if ch.isdigit():
            count = count * 10 + int(ch)
        # pause the outer string and its pending count
        elif ch == "[":
            stack.append((current, count))
            current, count = "", 0
        # repeat the inner string and attach it to the outer one
        elif ch == "]":
            prev, k = stack.pop()
            current = prev + current * k
        else:
            current += ch
    return current


assert decode_string("3[a]2[bc]") == "aaabcbc"
assert decode_string("3[a2[c]]") == "accaccacc"
assert decode_string("2[abc]3[cd]ef") == "abcabccdcdcdef"
assert decode_string("10[a]") == "a" * 10
print(decode_string("3[a2[c]]"))

accaccacc


### 1.1.9 [735. Asteroid Collision](https://leetcode.com/problems/asteroid-collision/) — ★★ · Medium

**Problem.** Asteroids move right (+) or left (−); colliding ones: smaller explodes, equal both explode. Final state?  
**Example.** `[5, 10, -5]` → `[5, 10]` (the -5 hits the 10 and explodes)  
**Spot it.** A left-mover only meets the most recent right-movers → stack.

**Ask first**
- Can moving-apart asteroids collide? → No — only `+` before `−`.

> 💡 **Idea.** Only a right-mover (on the stack) followed by a left-mover can crash. Keep survivors on a stack; while the new left-mover hits a smaller right-mover, pop it; equal sizes both explode.

**Brute force:** simulate time steps until nothing changes, O(n²).

**Steps**
1. For each asteroid, if it moves left and the top moves right, they collide.
2. Top smaller → pop it and keep colliding.
3. Equal → pop and destroy the incoming one too.
4. Top larger → incoming is destroyed. If it survives everything, push it.

**Cost:** Time O(n) (each asteroid pushed and popped at most once) · Space O(n)

**Likely follow-ups**
- **Explosions push debris that can trigger more collisions?** Same stack — a survivor just keeps fighting the next stack top (the while loop already chains).
- **Return *when* each one explodes (positions + speeds)?** That's Car Fleet II-style: right-to-left monotonic stack computing catch-up times, O(n).
- **O(1) extra space?** Use the input array as the stack with a write index.

In [ ]:
def asteroid_collision(asteroids):
    """Surviving asteroids after all collisions."""
    stack = []
    for a in asteroids:
        alive = True
        # only a left-mover meeting a right-mover collides
        while (alive and a < 0) and (stack and stack[-1] > 0):

            # top is smaller: it explodes, keep going
            if stack[-1] < -a:
                stack.pop()
                continue

            # equal: both explode
            if stack[-1] == -a:
                stack.pop()
            alive = False
        
        if alive:
            stack.append(a)
    return stack


assert asteroid_collision([5, 10, -5]) == [5, 10]
assert asteroid_collision([8, -8]) == []
assert asteroid_collision([10, 2, -5]) == [10]
assert asteroid_collision([-2, -1, 1, 2]) == [-2, -1, 1, 2]
print(asteroid_collision([5, 10, -5]))

[5, 10]


### 1.1.10 [1249. Minimum Remove to Make Valid Parentheses](https://leetcode.com/problems/minimum-remove-to-make-valid-parentheses/) — ★★ · Medium

**Problem.** Remove the fewest parentheses so the string is valid; return any valid result.  
**Example.** `"lee(t(c)o)de)"` → `"lee(t(c)o)de"`  
**Spot it.** Unmatched `)` are found left-to-right; unmatched `(` are what remain on the stack.

**Ask first**
- Any valid answer, or a specific one? → Any — so dropping the *first* unmatched `)` and *last* unmatched `(` is fine.

> 💡 **Idea.** Push the index of every `(`. A `)` with nothing to match gets marked for removal. Any `(` still on the stack at the end never closed — remove those too.

**Brute force:** try removing every subset of brackets, exponential (that's LC 301 for *all* answers).

**Steps**
1. Push the index of each `(`.
2. A `)` pops a partner, or is marked for removal if none exists.
3. After the scan, mark every index still on the stack.
4. Build the string without the marked indices.

**Cost:** Time O(n) · Space O(n)

**Likely follow-ups**
- **O(1) extra space (besides the output)?** Two passes with just counters: count `open` left→right skipping bad `)`, then remove the last `open` many `(` scanning right→left.
- **Several bracket types `()[]{}`?** Stack of `(char, index)`; a closer that doesn't match the top is removed. Needs O(n) space — counters can't track types.
- **Return *all* minimal-removal results (LC 301)?** BFS level by level (remove one char at a time) and stop at the first level with valid strings; or DFS with the removal counts precomputed.
- **Just the number of removals (LC 921)?** Unmatched `)` count + leftover `open` count.

In [ ]:
def min_remove_to_make_valid(s):
    """s with the fewest parentheses deleted to balance it."""
    remove, stack = set(), []
    for i, ch in enumerate(s):
        if ch == "(":
            stack.append(i)
        elif ch == ")":
            # matched closer, or one with no partner
            if stack:
                stack.pop()
            else:
                remove.add(i)
                
    # openers never closed
    remove.update(stack)
    return "".join(ch for i, ch in enumerate(s) if i not in remove)


assert min_remove_to_make_valid("lee(t(c)o)de)") == "lee(t(c)o)de"
assert min_remove_to_make_valid("a)b(c)d") == "ab(c)d"
assert min_remove_to_make_valid("))((") == ""
print(min_remove_to_make_valid("lee(t(c)o)de)"))

lee(t(c)o)de


### 1.1.11 [2390. Removing Stars From a String](https://leetcode.com/problems/removing-stars-from-a-string/) — ★ · Medium

**Problem.** Each `*` deletes the closest non-star character to its left; return the result.  
**Example.** `"leet**cod*e"` → `"lecoe"`  
**Spot it.** "Closest to the left" = top of a stack.

**Ask first**
- Is there always something to delete? → Yes, so no empty-stack check needed.

> 💡 **Idea.** A star deletes the closest letter to its left — that's a stack pop. Letters push, stars pop.

**Brute force:** find a star, delete it and the char before, rebuild the string — O(n²).

**Steps**
1. Push letters.
2. A `*` pops.
3. Join the stack.

**Cost:** Time O(n) · Space O(n)

**Likely follow-ups**
- **O(1) extra space with a char array?** Write pointer: letters are written at `w` and `w += 1`; a star does `w -= 1`. Answer is `arr[:w]`.
- **Stars delete the next letter to the right instead?** Scan right to left with the same stack / write-pointer idea.

In [12]:
def remove_stars(s):
    """s after applying star deletions."""
    stack = []
    for ch in s:
        # a star deletes the nearest kept letter
        if ch == "*":
            stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)


assert remove_stars("leet**cod*e") == "lecoe"
assert remove_stars("erase*****") == ""
print(remove_stars("leet**cod*e"))

lecoe


## 1.2 Monotonic stack

Keep the stack sorted. When a new element breaks the order, everything it pops has just found its answer. Each index is pushed and popped once → O(n).

| You want | Stack (bottom → top) | Pop while |
|---|---|---|
| next **greater** | decreasing | `top < x` |
| next **smaller** | increasing | `top > x` |
| previous greater / smaller | same stacks; read the new top *after* popping | — |

### 1.2.1 [739. Daily Temperatures](https://leetcode.com/problems/daily-temperatures/) — ★★ · Medium

**Problem.** For each day, how many days until a warmer temperature (0 if never)?  
**Example.** `[73, 74, 75, 71, 69, 72, 76, 73]` → `[1, 1, 4, 2, 1, 1, 0, 0]`  
**Spot it.** "Next greater element" → monotonic decreasing stack of indices.

**Ask first**
- Strictly warmer? → Yes, so pop on `<` (equal temps keep waiting).

> 💡 **Idea.** Keep a stack of days still waiting for a warmer day (temperatures decreasing). Each new day pops every colder waiting day — and tells it its answer.

**Brute force:** for each day scan forward, O(n²).

**Steps**
1. Keep a stack of indices still waiting.
2. For day `i`, while the top day is cooler, pop it and record `i − j`.
3. Push `i`.
4. Days left on the stack never get warmer (answer 0).

**Cost:** Time O(n) (each index pushed and popped once) · Space O(n)

**Likely follow-ups**
- **O(1) extra space (besides the answer)?** Go right to left; to find day i's answer, jump `j = i + 1`, then `j += ans[j]` while `temps[j] <= temps[i]` (and `ans[j] > 0`). Amortised O(n).
- **Temperatures wrap around (circular, LC 503)?** Loop `i` over `2n` indices with `i % n`; only push during the first lap.
- **Previous warmer day instead?** Same stack scanning left to right; after popping, the top is the previous warmer day.

In [13]:
def daily_temperatures(temperatures):
    """Days to wait for a warmer temperature."""
    out = [0] * len(temperatures)
    # indices still waiting; temperatures decrease toward the top
    stack = []
    for i, t in enumerate(temperatures):
        # today is the warmer day for every cooler waiting day
        while stack and temperatures[stack[-1]] < t:
            j = stack.pop()
            out[j] = i - j
        stack.append(i)
    return out


assert daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]) == [1, 1, 4, 2, 1, 1, 0, 0]
assert daily_temperatures([30, 60, 90]) == [1, 1, 0]
print(daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]))

[1, 1, 4, 2, 1, 1, 0, 0]


### 1.2.2 [901. Online Stock Span](https://leetcode.com/problems/online-stock-span/) — ★★ · Medium

**Problem.** For each new price, how many consecutive days (including today) had price ≤ today?  
**Example.** prices `100, 80, 60, 70, 60, 75, 85` → spans `1, 1, 1, 2, 1, 4, 6`  
**Spot it.** Previous greater element, online → monotonic stack of (price, span).

**Ask first**
- Span counts `<=` today and includes today? → Yes — pop on `<=`.

> 💡 **Idea.** Stack of `(price, span)` with prices decreasing. A new price swallows every smaller-or-equal price on top and adds their spans to its own.

**Brute force:** walk back through all past prices each call, O(n) per call.

**Steps**
1. Start today's span at 1.
2. While the top's price ≤ today's, pop it and add its span.
3. Push `(price, span)` and return the span.

**Cost:** Time Amortised O(1) per call · Space O(n)

**Likely follow-ups**
- **Only the last k days count (sliding window)?** Store indices; cap the span at k — or pop the bottom of a deque when it leaves the window.
- **Many stocks at once?** One stack per ticker in a dict; same amortised O(1) per call.
- **Can you also undo the last price?** Keep a history of what each call popped so it can be pushed back — costs O(n) memory.

In [ ]:
class StockSpanner:
    """Online stock span."""

    def __init__(self):
        # (price, days covered by it); prices decrease toward the top
        # pair(price, span)
        self.stack = []

    def next(self, price):
        span = 1
        # absorb every earlier day not above today
        while self.stack and self.stack[-1][0] <= price:
            span += self.stack.pop()[1]
        self.stack.append((price, span))
        return span


sp = StockSpanner()
out = [sp.next(p) for p in [100, 80, 60, 70, 60, 75, 85]]
assert out == [1, 1, 1, 2, 1, 4, 6]
print(out)

[1, 1, 1, 2, 1, 4, 6]


### 1.2.3 [853. Car Fleet](https://leetcode.com/problems/car-fleet/) — ★★ · Medium

**Problem.** Cars at `position[i]` with `speed[i]` drive to `target`; a faster car catching a slower one joins it. How many fleets arrive?  
**Example.** `target = 12, position = [10, 8, 0, 5, 3], speed = [2, 4, 1, 1, 3]` → `3`  
**Spot it.** Sort by position descending, compare arrival times → stack of fleet times.

**Ask first**
- Catching up exactly at the target? → Counts as the same fleet → use `<=`.

> 💡 **Idea.** Sort by position, closest to the target first. Compute each car's arrival time. A car behind that would arrive *sooner or equal* gets stuck behind → same fleet. A slower one starts a new fleet.

**Brute force:** simulate time in small steps, merging cars — slow and float-fragile.

**Steps**
1. Sort cars by position, nearest to target first.
2. Arrival time = `(target − pos) / speed`.
3. If the time is greater than the last fleet's time, it is a new fleet; otherwise it merges.
4. Count fleets.

**Cost:** Time O(n log n) (the sort) · Space O(n)

**Likely follow-ups**
- **When does each car first collide (Car Fleet II, LC 1776)?** Right-to-left monotonic stack; for each car pop cars it can't catch (or that collide before it gets there); time = `Δpos / Δspeed`. O(n).
- **Avoid floating point?** Compare times by cross-multiplying: `(t − p1) * s2 <= (t − p2) * s1`.
- **Return the fleets, not the count?** Group cars between each new-fleet leader.

In [ ]:
def car_fleet(target, position, speed):
    """Number of fleets reaching the target."""
    stack = []
    pairs = zip(position, speed)

    # closest car to the target first
    for pos, spd in sorted(pairs, reverse=True):
        time = (target - pos) / spd
        
        # slower than the fleet ahead: it can never catch up, new fleet
        if not stack or time > stack[-1]:
            stack.append(time)
    return len(stack)


assert car_fleet(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]) == 3
assert car_fleet(10, [3], [3]) == 1
assert car_fleet(100, [0, 2, 4], [4, 2, 1]) == 1
print(car_fleet(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]))

3


### 1.2.4 [84. Largest Rectangle in Histogram](https://leetcode.com/problems/largest-rectangle-in-histogram/) — ★★ · Hard

**Problem.** Largest rectangle area in a histogram.  
**Example.** `[2, 1, 5, 6, 2, 3]` → `10` (bars 5 and 6, width 2)  
**Spot it.** Each bar's widest rectangle extends to the previous and next *smaller* bars → monotonic increasing stack.

**Ask first**
- Width of each bar = 1? → Yes.

> 💡 **Idea.** For each bar, the widest rectangle at its height stretches until a shorter bar on each side. An increasing stack gives both: when a bar is popped, the new top is its left limit and the current index is its right limit.

**Brute force:** for each pair (i, j) track the min height, O(n²).

**Steps**
1. Keep indices with increasing heights; append a 0 sentinel to flush at the end.
2. When bar `i` is lower than the top, pop the top `j` (its next smaller is `i`).
3. Its previous smaller is the new top (or −1); width = `i − left − 1`.
4. Update the best area with `height[j] * width`; push `i`.

**Cost:** Time O(n) · Space O(n)

**Likely follow-ups**
- **Largest rectangle of 1s in a binary matrix (LC 85)?** Build a histogram per row (height of consecutive 1s above), run this on each row. O(rows·cols).

In [16]:
def largest_rectangle_area(heights):
    """Largest rectangle in a histogram."""
    # indices with increasing heights
    stack = []
    best = 0
    # the 0-height sentinel flushes every remaining bar
    for i, h in enumerate(heights + [0]):
        while stack and heights[stack[-1]] >= h:
            # i is the first lower bar to the right of the popped bar
            height = heights[stack.pop()]
            # the new top is the first lower bar to its left
            left = stack[-1] if stack else -1
            best = max(best, height * (i - left - 1))
        stack.append(i)
    return best


assert largest_rectangle_area([2, 1, 5, 6, 2, 3]) == 10
assert largest_rectangle_area([2, 4]) == 4
assert largest_rectangle_area([1, 1, 1, 1]) == 4
print(largest_rectangle_area([2, 1, 5, 6, 2, 3]))

10


### 1.2.5 [402. Remove K Digits](https://leetcode.com/problems/remove-k-digits/) — ★ · Medium

**Problem.** Remove `k` digits from a number string to make it as small as possible.  
**Example.** `num = "1432219", k = 3` → `"1219"`  
**Spot it.** Lexicographically smallest → greedy monotonic increasing stack.

**Ask first**
- Leading zeros in the result? → Strip them; empty result → `"0"`.

> 💡 **Idea.** To make the number small, the left digits matter most. Keep an increasing stack: while you still can delete and the top is bigger than the new digit, pop it. Then trim leftovers from the end and strip leading zeros.

**Brute force:** try deleting every combination of k digits, C(n, k).

**Steps**
1. For each digit, while `k > 0` and the top is larger, pop it and decrement `k`.
2. Push the digit.
3. If `k` remains, drop digits from the end (the stack is increasing).
4. Strip leading zeros; empty → `"0"`.

**Cost:** Time O(n) · Space O(n)

**Likely follow-ups**
- **Largest number instead?** Flip it: pop while the top is *smaller* than the new digit. No zero stripping needed.
- **Keep k digits (pick a subsequence) from two arrays (LC 321)?** Best i digits from A + best k−i from B with this stack, merge greedily, try all i.
- **Smallest subsequence with all distinct letters once (LC 316/1081)?** Same stack, but only pop if the popped letter appears again later; skip letters already in the stack.

In [17]:
def remove_kdigits(num, k):
    """Smallest number after deleting k digits."""
    stack = []
    for d in num:
        # a larger digit before a smaller one is worth deleting
        while k and stack and stack[-1] > d:
            stack.pop()
            k -= 1
        stack.append(d)
    # remaining deletions come off the (largest) end
    if k:
        stack = stack[:-k]
    return "".join(stack).lstrip("0") or "0"


assert remove_kdigits("1432219", 3) == "1219"
assert remove_kdigits("10200", 1) == "200"
assert remove_kdigits("10", 2) == "0"
print(remove_kdigits("1432219", 3))

1219


---
# 2) Binary Search

**The gist.** If a yes/no question flips from "no" to "yes" exactly once along a range, check the middle and throw away the half that can't hold the flip.

> **Think of it like…** "higher / lower" guessing between 1 and 100 — seven guesses always do it, because each one halves what's left.

**Search on the answer.** For "the minimum speed / capacity / day such that it works": the answer is an integer in some range, and "it works" is monotone (if speed `v` works, faster works too). Binary-search the range and test each guess with an O(n) greedy check.

**Spot it:** sorted / rotated sorted / "O(log n)" · first / last position, insert position · "minimum X such that…" · huge input range but a cheap check · "k-th smallest" where you can *count* elements ≤ v.

**Classic bugs:** mixing templates (`lo <= hi` with `hi = mid` → infinite loop) · a start range that doesn't contain the answer (capacity starts at `max(weights)`, not 1) · a predicate that isn't monotone · ceil division is `(a + b − 1) // b`.

The cheatsheet below has the exact signs for each template.

## Cheatsheet: which sign do I write?

Two things decide every line: **what the loop does when `lo == hi`**, and **whether you keep `mid`**.

**Two matching pairs. Never mix them.**

| Loop | `lo` / `hi` after checking `mid` | Why |
|---|---|---|
| `while lo <= hi` | `lo = mid + 1`, `hi = mid - 1` | both ends are real candidates; `mid` was checked, so drop it on either side |
| `while lo < hi` | `lo = mid + 1`, `hi = mid` | the loop stops at one candidate; `mid` might be the answer, so keep it |

So: `lo = mid + 1` always. `hi = mid - 1` goes with `<=`, and `hi = mid` goes with `<`.

**Which template for which question**

| I want… | Loop | On a hit / true | Else | Return | Questions |
|---|---|---|---|---|---|
| **Exact match** | `lo <= hi` | `nums[mid] == target` → `return mid` | `<` → `lo = mid + 1`, `>` → `hi = mid - 1` | `-1` (or `lo` for insert position) | 704, 35, 74, 33, 81, 162 (the "hit" is: no neighbour is bigger) |
| **First true** (lower bound, smallest answer) | `lo <= hi` | `hi = mid - 1` | `lo = mid + 1` | `lo` (can be `n`) | 278, 69, 2300, 875, 1011, 378 |
| **Last true** (largest answer) | `lo <= hi` | `lo = mid + 1` | `hi = mid - 1` | `hi` (can be `-1`) | same code, flipped |
| **First / last copy** | `lo <= hi` | `found = mid`, then `hi = mid - 1` (first) or `lo = mid + 1` (last) | move toward the target | `found` | 34 |
| **Shrink to one** (parity, rotated min, windows) | `lo < hi` | `hi = mid` | `lo = mid + 1` | `lo` (or `nums[lo]`) | 540, 658, 153 |

**Skeletons**

```python
# exact match                      # first true                      # shrink to one
lo, hi = 0, n - 1                  lo, hi = 0, n - 1                 lo, hi = 0, n - 1
while lo <= hi:                    while lo <= hi:                   while lo < hi:
    mid = (lo + hi) // 2               mid = (lo + hi) // 2              mid = (lo + hi) // 2
    if nums[mid] == target:            if predicate(mid):                if keep_mid(mid):
        return mid                         hi = mid - 1                      hi = mid
    elif nums[mid] < target:           else:                             else:
        lo = mid + 1                       lo = mid + 1                      lo = mid + 1
    else:                          return lo                         return lo
        hi = mid - 1
return -1
```

**Where does `hi` start?** At the **last value that could be the answer**.

| Search space | `lo, hi` |
|---|---|
| array indices | `0, n - 1` |
| versions `1..n` | `1, n` |
| answer range (speed, capacity) | the smallest and largest feasible answer, e.g. `1, max(piles)` |
| 2D matrix as one list | `0, rows * cols - 1` |

**`<` or `<=` in the check?**

- first index `>= x` (lower bound): `nums[mid] >= x` → go left
- first index `> x` (upper bound): `nums[mid] > x` → go left
- "feasible" checks usually say `<=` (e.g. `hours <= h`, `days <= limit`): true means "this works, try smaller"

**Before you return**

- After a **first-true** loop, `lo` can be `n` (nothing was true). Check `lo < len(nums)` before reading `nums[lo]`.
- After a **last-true** loop, `hi` can be `-1`.
- Reading `nums[mid + 1]`? Use `lo < hi`: then `mid <= hi - 1`, so `mid + 1` is always valid.
- Quick tests: empty list, one element, target smaller than everything, target larger than everything.


## 2.1 Classic binary search

Same loop every time; only the predicate changes. Write it as a sentence first ("`nums[i] >= target`", "`is_bad(v)`", "`mid * mid > x`").

### 2.1.1 [704. Binary Search](https://leetcode.com/problems/binary-search/) — ★★ · Easy

**Problem.** Index of `target` in a sorted array, or −1.  
**Example.** `nums = [-1, 0, 3, 5, 9, 12], target = 9` → `4`  
**Spot it.** Sorted + lookup → binary search.

**Ask first**
- Must it be O(log n)? → Yes, so no linear scan.

> 💡 **Idea.** Check the middle; drop the half that can't contain the target.

**Brute force:** linear scan, O(n).

**Steps**
1. `lo = 0`, `hi = n - 1`; loop `while lo <= hi`.
2. If `nums[mid] == target`, return `mid`.
3. If `nums[mid] < target`, the target is to the right (`lo = mid + 1`); otherwise it is to the left (`hi = mid - 1`).
4. If the range becomes empty (`lo > hi`), return −1.

**Cost:** Time O(log n) (the range halves each step) · Space O(1)

In [ ]:
def search(nums, target):
    """Index of target in sorted nums, or -1."""
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        # found it
        if nums[mid] == target:
            return mid
        # mid is too small: the target is to its right
        elif nums[mid] < target:
            lo = mid + 1
        # mid is too big: the target is to its left
        else:
            hi = mid - 1
    # range is empty: target is not in nums
    return -1


assert search([-1, 0, 3, 5, 9, 12], 9) == 4
assert search([-1, 0, 3, 5, 9, 12], 2) == -1
print(search([-1, 0, 3, 5, 9, 12], 9))

4


### 2.1.2 [35. Search Insert Position](https://leetcode.com/problems/search-insert-position/) — ★★ · Easy

**Problem.** Index of `target`, or where it would be inserted to keep order.  
**Example.** `[1, 3, 5, 6], target = 5` → `2`; `target = 2` → `1`  
**Spot it.** Insertion point in a sorted array → binary search, and `lo` is the answer when the target is absent.

**Ask first**
- Insert position can be `n`? → Yes, past the end — so `hi` starts at `n` (or return `lo`).

> 💡 **Idea.** The insert position is the first index whose value is ≥ target (lower bound).

**Brute force:** scan until a value ≥ target, O(n).

**Steps**
1. Run the basic loop: `lo = 0`, `hi = n - 1`, `while lo <= hi`; return `mid` on a match.
2. If `nums[mid] < target`, `lo = mid + 1`; else `hi = mid - 1`.
3. On no match, return `lo` (may equal `n`: insert at the end).

**Cost:** Time O(log n) · Space O(1)

In [19]:
def search_insert(nums, target):
    """Index of target, or where it would be inserted."""
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        elif nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    # loop ended with lo = hi + 1; lo may be len(nums): inserting after everything is allowed
    return lo


assert search_insert([1, 3, 5, 6], 5) == 2
assert search_insert([1, 3, 5, 6], 2) == 1
assert search_insert([1, 3, 5, 6], 7) == 4
assert search_insert([1, 3, 5, 6], 0) == 0
print(search_insert([1, 3, 5, 6], 2))

1


### 2.1.3 [278. First Bad Version](https://leetcode.com/problems/first-bad-version/) — ★★ · Easy

**Problem.** Versions `1..n`; after the first bad one, all are bad. Find it with few API calls.  
**Example.** `n = 5`, versions 4 and 5 are bad → `4`  
**Spot it.** Monotone predicate `is_bad(v)` → first-true binary search.

**Ask first**
- Goal is the fewest API calls? → Yes → binary search, not a scan.

> 💡 **Idea.** Versions go good…good bad…bad. Binary-search the first bad one.

**Brute force:** call `is_bad` on every version, O(n) calls.

**Steps**
1. `lo = 1`, `hi = n`; loop `while lo <= hi`.
2. `is_bad(mid)` → an earlier version might be bad too, so `hi = mid - 1`; else `lo = mid + 1`.
3. The loop ends with `lo = hi + 1`, and `lo` is the first bad version.

**Cost:** Time O(log n) API calls · Space O(1)

In [20]:
def first_bad_version(n, is_bad):
    """First version for which is_bad is True."""
    lo, hi = 1, n
    while lo <= hi:
        mid = (lo + hi) // 2
        # mid is bad: it may be the first, but an earlier one might be bad too
        if is_bad(mid):
            hi = mid - 1
        else:
            lo = mid + 1
    # loop ended with lo = hi + 1: lo is the first bad version
    return lo


assert first_bad_version(5, lambda v: v >= 4) == 4
assert first_bad_version(1, lambda v: v >= 1) == 1
print(first_bad_version(2**31 - 1, lambda v: v >= 1702766719))

1702766719


### 2.1.4 [34. Find First and Last Position of Element in Sorted Array](https://leetcode.com/problems/find-first-and-last-position-of-element-in-sorted-array/) — ★★ · Medium

**Problem.** `[first, last]` index of `target` in a sorted array, or `[-1, -1]`.  
**Example.** `nums = [5, 7, 7, 8, 8, 10], target = 8` → `[3, 4]`  
**Spot it.** Sorted, first and last position → binary search that keeps going after a match, once leaning left and once leaning right.

**Ask first**
- Not found? → `[-1, -1]` — check the first index is in range and actually equals target.

> 💡 **Idea.** Run binary search twice. On a match, don't stop — keep going left to find the first copy, or right to find the last.

**Brute force:** find any copy, then walk left and right — O(n) when there are many copies.

**Steps**
1. `first = bin_search(left_bias=True)`: on a match, record it and go left (`hi = mid - 1`).
2. If `first == -1`, return `[-1, -1]`.
3. `last = bin_search(left_bias=False, lo=first)`: on a match, record it and go right (`lo = mid + 1`). Starting at `first` skips the part that cannot hold the last copy.

**Cost:** Time O(log n) (two searches) · Space O(1)

**Likely follow-ups**
- **Count how many times target appears?** `upper_bound − lower_bound`, O(log n). 0 if absent.
- **Why not find one copy and expand?** All-equal arrays make it O(n); two boundary searches stay O(log n).
- **Values come as a stream of queries?** Same — each query is two O(log n) searches; precompute nothing.

In [ ]:
def search_range(nums, target):
    """First and last index of target."""

    def bin_search(left_bias, lo=0):
        """Index of a match leaning left (first copy) or right (last copy), else -1."""
        hi = len(nums) - 1
        found = -1
        while lo <= hi:
            mid = (lo + hi) // 2
            if nums[mid] < target:
                lo = mid + 1
            elif nums[mid] > target:
                hi = mid - 1
            else:
                # a match: remember it, then keep looking on one side for a better one
                found = mid
                if left_bias:
                    hi = mid - 1
                else:
                    lo = mid + 1
        return found

    first = bin_search(left_bias=True)
    if first == -1:
        return [-1, -1]
    # the last copy cannot be left of the first copy
    return [first, bin_search(left_bias=False, lo=first)]


assert search_range([5, 7, 7, 8, 8, 10], 8) == [3, 4]
assert search_range([5, 7, 7, 8, 8, 10], 6) == [-1, -1]
assert search_range([], 0) == [-1, -1]
print(search_range([5, 7, 7, 8, 8, 10], 8))

[3, 4]


### 2.1.5 [74. Search a 2D Matrix](https://leetcode.com/problems/search-a-2d-matrix/) — ★★ · Medium

**Problem.** Rows sorted, and each row starts after the previous row ends. Is `target` present?  
**Example.** `matrix = [[1,3,5,7],[10,11,16,20],[23,30,34,60]], target = 3` → `True`  
**Spot it.** The matrix is one sorted array of length `m·n` in disguise.

**Ask first**
- Does each row start after the previous row ends? → Yes. If not, it's LC 240 (staircase search).

> 💡 **Idea.** Each row starts above the last row's end, so it's one sorted list of `m·n` items. Binary-search index `i` and read `matrix[i // n][i % n]`.

**Brute force:** scan every cell, O(m·n).

**Steps**
1. Search `k` in `[0, m·n − 1]`.
2. Read `matrix[k // cols][k % cols]`.
3. Equal → found; smaller → go right; larger → go left.

**Cost:** Time O(log(mn)) · Space O(1)

**Likely follow-ups**
- **Rows sorted but don't connect (each row only sorted on its own)?** Binary-search each row (O(m log n)); if columns are also sorted, use the staircase from the top-right (O(m + n), LC 240).
- **Two searches instead of one?** Find the row with a binary search on first elements, then search inside it: O(log m + log n) — same total.

In [22]:
def search_matrix(matrix, target):
    """True if target is in the row-major sorted matrix."""
    rows, cols = len(matrix), len(matrix[0])
    lo, hi = 0, rows * cols - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        # flat index -> (row, column)
        value = matrix[mid // cols][mid % cols]
        if value == target:
            return True
        if value < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return False


m = [[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]]
assert search_matrix(m, 3)
assert not search_matrix(m, 13)
print(search_matrix(m, 34))

True


### 2.1.6 [240. Search a 2D Matrix II](https://leetcode.com/problems/search-a-2d-matrix-ii/) — ★★ · Medium

**Problem.** Rows and columns sorted independently. Is `target` present?  
**Example.** `matrix = [[1,4,7],[2,5,8],[3,6,9]], target = 5` → `True`  
**Spot it.** Start at the top-right corner: it splits the remaining matrix like a BST root.

**Ask first**
- Rows and columns sorted, but not as one list? → Yes → staircase, not a single binary search.

> 💡 **Idea.** Start at the top-right. Too big → move left (the whole column below is bigger). Too small → move down (the whole row to the left is smaller). Each step kills a row or a column.

**Brute force:** scan everything, O(m·n); binary-search each row, O(m log n).

**Steps**
1. Start at the top-right cell.
2. Equal → found.
3. Too big → move left (this column is all too big below).
4. Too small → move down (this row is all too small to the left).

**Cost:** Time O(m + n) (each step removes a row or a column) · Space O(1)

**Likely follow-ups**
- **Count elements ≤ x?** Same staircase from the bottom-left: when `matrix[r][c] <= x`, add `r + 1` and move right; else move up. O(m + n).
- **k-th smallest (LC 378)?** Binary-search the value range and use that count as the check: O((m + n) log(range)).
- **Why not binary search?** A single binary search doesn't work since the matrix isn't one sorted list; divide-and-conquer exists but the staircase is simpler and O(m + n).

In [ ]:
def search_matrix_ii(matrix, target):
    """True if target is in a row- and column-sorted matrix."""
    # top-right corner: left is smaller, down is larger
    r, c = 0, len(matrix[0]) - 1
    
    while r < len(matrix) and c >= 0:
        value = matrix[r][c]
        if value == target:
            return True
        # this column is too big from here down
        if value > target:
            c -= 1
        # this row is too small from here left
        else:
            r += 1
    return False


m = [
    [1, 4, 7, 11, 15],
    [2, 5, 8, 12, 19],
    [3, 6, 9, 16, 22],
    [10, 13, 14, 17, 24],
    [18, 21, 23, 26, 30],
]
assert search_matrix_ii(m, 5)
assert not search_matrix_ii(m, 20)
print(search_matrix_ii(m, 5))

True


### 2.1.7 [69. Sqrt(x)](https://leetcode.com/problems/sqrtx/) — ★★ · Easy

**Problem.** Integer square root (floor), no built-in power functions.  
**Example.** `8` → `2` (because 2 × 2 = 4 and 3 × 3 = 9 is too big)  
**Spot it.** Largest `r` with `r² ≤ x` → binary search on the answer.

**Ask first**
- Round down? → Yes, floor.

> 💡 **Idea.** Find the largest `m` with `m * m <= x` — a last-true binary search on the answer.

**Brute force:** try 1, 2, 3, … until the square passes x, O(√x).

**Steps**
1. Range `[0, x]`.
2. Predicate `mid * mid > x`.
3. Return `lo − 1`.

**Cost:** Time O(log x) · Space O(1)

In [24]:
def my_sqrt(x):
    """floor(sqrt(x))."""
    lo, hi = 0, x
    while lo <= hi:
        mid = (lo + hi) // 2
        # first r whose square overshoots x
        if mid * mid > x:
            hi = mid - 1
        else:
            lo = mid + 1
    # step back to the last r with r*r <= x
    return lo - 1


assert my_sqrt(4) == 2
assert my_sqrt(8) == 2
assert my_sqrt(0) == 0
assert my_sqrt(2147395599) == 46339
print(my_sqrt(8))

2


### 2.1.8 [162. Find Peak Element](https://leetcode.com/problems/find-peak-element/) — ★★ · Medium

**Problem.** Return any index whose value is greater than its neighbours (`nums[-1] = nums[n] = −∞`).  
**Example.** `[1, 2, 3, 1]` → `2` (the 3 is bigger than both neighbours)  
**Spot it.** Binary search on the slope: move toward the bigger neighbour.

**Ask first**
- Can neighbours be equal? → No — that's what guarantees a peak (with plateaus binary search can fail).

> 💡 **Idea.** If `nums[mid] < nums[mid + 1]`, you're on an uphill — a peak must be to the right. Otherwise one is at `mid` or to its left.

**Brute force:** scan for any `nums[i] > neighbours`, O(n).

**Steps**
1. Range `[0, n − 1]`; a missing neighbour at the edge is never bigger (`mid > 0`, `mid < n − 1`).
2. Left neighbour bigger → peak on the left: `hi = mid − 1`.
3. Right neighbour bigger → peak on the right: `lo = mid + 1`.
4. Neither is bigger → return `mid`.

**Cost:** Time O(log n) · Space O(1)

**Likely follow-ups**
- **2D peak (LC 1901)?** Binary-search the rows: take the max of the middle row; if the cell below is bigger, go down, else go up. O(n log m).
- **Why does a peak always exist on the uphill side?** Edges count as −∞. Going right from an uphill step, values either keep rising to the end (last is a peak) or drop at some point (that's a peak).
- **Find all peaks?** Needs O(n) — scan.

In [25]:
def find_peak_element(nums):
    """Index of some local maximum."""
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        # the edges count as -infinity, so a missing neighbour is never bigger
        left_bigger = mid > 0 and nums[mid - 1] > nums[mid]
        right_bigger = mid < len(nums) - 1 and nums[mid + 1] > nums[mid]
        # bigger neighbour on the left: a peak must exist on the left
        if left_bigger:
            hi = mid - 1
        # bigger neighbour on the right: a peak must exist on the right
        elif right_bigger:
            lo = mid + 1
        # no neighbour is bigger: mid is a peak
        else:
            return mid


assert find_peak_element([1, 2, 3, 1]) == 2
assert find_peak_element([1, 2, 1, 3, 5, 6, 4]) in (1, 5)
print(find_peak_element([1, 2, 1, 3, 5, 6, 4]))

5


### 2.1.9 [540. Single Element in a Sorted Array](https://leetcode.com/problems/single-element-in-a-sorted-array/) — ★ · Medium

**Problem.** Sorted array where every element appears twice except one; find it in O(log n).  
**Example.** `[1, 1, 2, 3, 3, 4, 4, 8, 8]` → `2`  
**Spot it.** Before the single element, pairs start at even indices; after it, at odd indices.

**Ask first**
- Must it be O(log n)? → Yes. Otherwise XOR everything (O(n)).

> 💡 **Idea.** Before the single element, pairs start at even indices; after it, pairs start at odd ones. Check `mid` (made even) against `mid + 1` to see which side you're on.

**Brute force:** XOR all, or check neighbours, O(n).

**Steps**
1. Make `mid` even (subtract 1 if odd).
2. If `nums[mid] == nums[mid + 1]`, pairs are intact up to here → `lo = mid + 2`.
3. Otherwise the single element is at `mid` or left → `hi = mid`.
4. Return `nums[lo]`.

**Cost:** Time O(log n) · Space O(1)

**Likely follow-ups**
- **Array isn't sorted?** XOR of all elements, O(n), O(1).
- **Every other element appears three times (LC 137)?** Count bits mod 3 for each of the 32 positions.
- **Two singles, rest pairs (LC 260)?** XOR all → `a ^ b`; split by its lowest set bit into two groups and XOR each.

In [ ]:
def single_non_duplicate(nums):
    lo, hi = 0, len(nums) - 1

    while lo <= hi:
        mid = (lo + hi) // 2
        same_as_left = mid > 0 and nums[mid - 1] == nums[mid]
        same_as_right = mid < len(nums) - 1 and nums[mid + 1] == nums[mid]

        # no twin on either side: mid is the odd one
        if not same_as_left and not same_as_right:
            return nums[mid]

        # where does mid's pair sit?
        if same_as_left:
            start, end = mid - 1, mid
        else:
            start, end = mid, mid + 1

        # start = how many numbers are to the left of the pair
        # left side is whole pairs: odd one is on the right
        if start % 2 == 0:
            lo = end + 1
        # left side has a stray: odd one is on the left
        else:
            hi = start - 1
      

assert single_non_duplicate([1, 1, 2, 3, 3, 4, 4, 8, 8]) == 2
assert single_non_duplicate([3, 3, 7, 7, 10, 11, 11]) == 10
print(single_non_duplicate([1, 1, 2, 3, 3, 4, 4, 8, 8]))

2


### 2.1.10 [658. Find K Closest Elements](https://leetcode.com/problems/find-k-closest-elements/) — ★★ · Medium

**Problem.** The `k` values closest to `x` in a sorted array, returned sorted (ties → smaller).  
**Example.** `arr = [1, 2, 3, 4, 5], k = 4, x = 3` → `[1, 2, 3, 4]`  
**Spot it.** The answer is a window of length `k` → binary search the window's left edge.

**Ask first**
- Distance ties? → The smaller value wins (that's why it's `>` not `>=`).
- Output sorted? → Yes, ascending — the block is already sorted.

> 💡 **Idea.** The answer is a block of k consecutive numbers. Binary-search its left edge: if `x − arr[mid] > arr[mid + k] − x`, the block should start further right.

**Brute force:** sort by `(|v − x|, v)` and take k, then sort them, O(n log n).

**Steps**
1. Search `left` in `[0, n − k]`.
2. If `x − arr[mid] > arr[mid + k] − x`, the left end is worse → shift right.
3. Otherwise keep `mid` (ties prefer the left).
4. Return `arr[lo : lo + k]`.

**Cost:** Time O(log(n − k) + k) · Space O(1) besides the output

**Likely follow-ups**
- **Array is unsorted?** Max-heap of size k on `(distance, value)` → O(n log k); or quickselect by distance, O(n) average. Sort the k results at the end.
- **Simpler O(log n + k)?** Binary-search the insert position of x, then expand two pointers outward k times.
- **Stream of numbers, keep the k closest to x?** Max-heap of size k keyed by distance.

In [27]:
def find_closest_elements(arr, k, x):
    """k closest values to x, sorted."""
    # candidate window starts
    lo, hi = 0, len(arr) - k
    while lo < hi:
        mid = (lo + hi) // 2
        # left end farther than the next element on the right: slide right
        if x - arr[mid] > arr[mid + k] - x:
            lo = mid + 1
        else:
            hi = mid
    return arr[lo : lo + k]


assert find_closest_elements([1, 2, 3, 4, 5], 4, 3) == [1, 2, 3, 4]
assert find_closest_elements([1, 1, 2, 3, 4, 5], 4, -1) == [1, 1, 2, 3]
print(find_closest_elements([1, 2, 3, 4, 5], 4, 3))

[1, 2, 3, 4]


### 2.1.11 [2300. Successful Pairs of Spells and Potions](https://leetcode.com/problems/successful-pairs-of-spells-and-potions/) — ★ · Medium

**Problem.** For each spell, count potions with `spell × potion ≥ success`.  
**Example.** `spells = [5, 1, 3], potions = [1, 2, 3, 4, 5], success = 7` → `[4, 0, 3]`  
**Spot it.** Sort potions once; each spell needs a threshold lookup.

**Ask first**
- Products > 32 bits? → Yes (up to 10¹⁰) — fine in Python; use `long` elsewhere, or compare with division.

> 💡 **Idea.** Sort potions. For each spell, the weakest potion that works is `ceil(success / spell)`; binary-search it and count everything to its right.

**Brute force:** try every pair, O(n·m).

**Steps**
1. Sort potions.
2. For each spell, `need = ceil(success / s)`.
3. Lower bound of `need` in potions.
4. Count = `m − lower_bound`.

**Cost:** Time O((n + m) log m) (sort plus one search per spell) · Space O(m) for the sorted copy

**Likely follow-ups**
- **Without a binary search per spell?** Sort spells too (remember original indices); as spells get stronger, the threshold pointer in potions only moves left → two pointers, O(n log n + m log m).
- **Potions get added between queries?** Keep potions in a sorted structure (bisect.insort / balanced BST).

In [ ]:
def successful_pairs(spells, potions, success):
    """Number of successful potions per spell."""
    potions = sorted(potions)
    m = len(potions)
    out = []

    for s in spells:
        # weakest potion that still works (ceil division)
        # math.ceil(success / s)
        need = (success + s - 1) // s

        lo, hi = 0, m - 1
        while lo <= hi:
            mid = (lo + hi) // 2
            if potions[mid] >= need:
                hi = mid - 1
            else:
                lo = mid + 1
        # every potion from lo onward works
        out.append(m - lo)

    return out


assert successful_pairs([5, 1, 3], [1, 2, 3, 4, 5], 7) == [4, 0, 3]
assert successful_pairs([3, 1, 2], [8, 5, 8], 16) == [2, 0, 2]
print(successful_pairs([5, 1, 3], [1, 2, 3, 4, 5], 7))

[4, 0, 3]


## 2.2 Rotated arrays

A rotated sorted array is two sorted runs. At any `mid`, at least one side is sorted — compare `nums[mid]` with an end to see which, then ask whether the target can be in that sorted side.

### 2.2.1 [153. Find Minimum in Rotated Sorted Array](https://leetcode.com/problems/find-minimum-in-rotated-sorted-array/) — ★★★ · Medium

**Problem.** Minimum of a rotated sorted array of distinct values, O(log n).  
**Example.** `[3, 4, 5, 1, 2]` → `1`  
**Spot it.** Compare `mid` with the right end to know which half is sorted.

**Ask first**
- Duplicates? → Not here. With duplicates (LC 154) worst case becomes O(n).

> 💡 **Idea.** Compare `nums[mid]` with `nums[hi]`. Bigger → the drop (minimum) is to the right. Smaller or equal → the minimum is at `mid` or left.

**Brute force:** scan for the min, O(n).

**Steps**
1. `lo = 0`, `hi = n − 1`.
2. `nums[mid] > nums[hi]` → `lo = mid + 1`.
3. Otherwise → `hi = mid`.
4. Return `nums[lo]`.

**Cost:** Time O(log n) · Space O(1)

**Likely follow-ups**
- **With duplicates (LC 154)?** When `nums[mid] == nums[hi]`, you can't tell which side — do `hi -= 1`. Worst case O(n) (e.g. `[1,1,1,0,1]`).
- **How many times was it rotated?** The index of the minimum.
- **Why compare with `nums[hi]`, not `nums[lo]`?** Comparing with `lo` breaks when the array isn't rotated at all.

In [29]:
def find_min(nums):
    """Minimum of a rotated sorted array."""
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        # mid sits in the upper run: the drop is to the right
        if nums[mid] > nums[hi]:
            lo = mid + 1
        # mid..hi is sorted: the minimum is mid or earlier
        else:
            hi = mid
    return nums[lo]


assert find_min([3, 4, 5, 1, 2]) == 1
assert find_min([4, 5, 6, 7, 0, 1, 2]) == 0
assert find_min([11, 13, 15, 17]) == 11
print(find_min([4, 5, 6, 7, 0, 1, 2]))

0


### 2.2.2 [33. Search in Rotated Sorted Array](https://leetcode.com/problems/search-in-rotated-sorted-array/) — ★★★ · Medium

**Problem.** Index of `target` in a rotated sorted array of distinct values, O(log n).  
**Example.** `nums = [4, 5, 6, 7, 0, 1, 2], target = 0` → `4`  
**Spot it.** One half around `mid` is always sorted → check whether `target` lies in that half.

**Ask first**
- Distinct values? → Yes. With duplicates it's LC 81 and worst case O(n).

> 💡 **Idea.** At any `mid`, one half is sorted. If the target sits inside that sorted half's range, go there; otherwise go to the other half.

**Brute force:** linear scan, O(n).

**Steps**
1. If `nums[mid] == target`, return `mid`.
2. If `nums[lo] <= nums[mid]`, the left half is sorted: go left iff `nums[lo] <= target < nums[mid]`.
3. Otherwise the right half is sorted: go right iff `nums[mid] < target <= nums[hi]`.
4. Not found → −1.

**Cost:** Time O(log n) · Space O(1)

**Likely follow-ups**
- **With duplicates (LC 81)?** When `nums[lo] == nums[mid] == nums[hi]`, shrink both ends by one. Worst case O(n).
- **Find the rotation point first, then search?** Also O(log n): find the min index (LC 153), then normal binary search on the correct half.
- **Rotated array, but you don't know the length (infinite)?** Double `hi` until you pass the target or hit the end, then search.

In [ ]:
def search_rotated(nums, target):
    """Index of target in a rotated sorted array, or -1."""
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        
        # left half is sorted
        if nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        # right half is sorted
        else:
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return -1


assert search_rotated([4, 5, 6, 7, 0, 1, 2], 0) == 4
assert search_rotated([4, 5, 6, 7, 0, 1, 2], 3) == -1
assert search_rotated([1], 0) == -1
assert search_rotated([3, 1], 1) == 1
print(search_rotated([4, 5, 6, 7, 0, 1, 2], 0))

4


### 2.2.3 [81. Search in Rotated Sorted Array II](https://leetcode.com/problems/search-in-rotated-sorted-array-ii/) — ★★ · Medium

**Problem.** Same as above, but duplicates are allowed; return whether `target` exists.  
**Example.** `nums = [2, 5, 6, 0, 0, 1, 2], target = 0` → `True`  
**Spot it.** Duplicates can hide which half is sorted → when `nums[lo] == nums[mid] == nums[hi]`, shrink both ends.

**Ask first**
- Return a bool? → Yes, just found or not.

> 💡 **Idea.** Same as 33, plus one case: if `nums[lo] == nums[mid] == nums[hi]` you can't tell which half is sorted, so shrink `lo += 1, hi -= 1` and try again.

**Brute force:** `target in nums`, O(n).

**Steps**
1. If `nums[mid] == target`, return `True`.
2. If `nums[lo] == nums[mid] == nums[hi]`, move `lo` and `hi` inward by one.
3. Otherwise apply the distinct-values logic.

**Cost:** Time O(log n) average, O(n) worst (e.g. all values equal) · Space O(1)

**Likely follow-ups**
- **Why is the worst case O(n)?** An array like `[1,1,1,1,0,1,1]`: every step might hit equal ends and only shrink by one.
- **Return the first index of target?** After finding, you'd need the rotation point; with duplicates that's O(n) worst case anyway.

In [ ]:
def search_rotated_ii(nums, target):
    """True if target is in a rotated sorted array with duplicates."""
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return True
        
        # cannot tell which half is sorted: trim both ends
        if nums[lo] == nums[mid] == nums[hi]:
            lo, hi = lo + 1, hi - 1
            
        # left half sorted
        elif nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1

        # right half sorted
        else:
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return False


assert search_rotated_ii([2, 5, 6, 0, 0, 1, 2], 0)
assert not search_rotated_ii([2, 5, 6, 0, 0, 1, 2], 3)
assert search_rotated_ii([1, 0, 1, 1, 1], 0)
print(search_rotated_ii([2, 5, 6, 0, 0, 1, 2], 0))

True


## 2.3 Search on the answer

Recipe: (1) what are you searching for (speed, capacity, day, value)? (2) its smallest and largest possible values (3) `feasible(v)`, usually a greedy O(n) scan (4) check it's monotone (5) run the first-true template.

### 2.3.1 [875. Koko Eating Bananas](https://leetcode.com/problems/koko-eating-bananas/) — ★★ · Medium

**Problem.** Minimum eating speed `k` to finish all piles within `h` hours (one pile per hour max).  
**Example.** `piles = [3, 6, 7, 11], h = 8` → `4`  
**Spot it.** "Minimum speed such that feasible" + monotone feasibility → binary search on `k`.

**Ask first**
- One pile per hour max? → Yes — leftover time in an hour is wasted, hence the `ceil`.

> 💡 **Idea.** If Koko can finish at speed `k`, she can at any faster speed. Binary-search the smallest `k` in `[1, max(piles)]` where `sum(ceil(p / k)) <= h`.

**Brute force:** try every speed 1, 2, 3… until it works, O(max · n).

**Steps**
1. Speeds range over `[1, max(piles)]`.
2. Hours at speed `k` = `sum(ceil(p / k))`.
3. First speed with hours ≤ `h` via the template.

**Cost:** Time O(n log max) (log max candidates, O(n) check each) · Space O(1)

**Likely follow-ups**
- **Prove the check is monotone.** Each `ceil(p / k)` can only shrink as k grows, so total hours never go up.
- **Same pattern elsewhere?** Min days to make m bouquets (LC 1482), min time to complete trips (LC 2187), split array largest sum (LC 410).
- **Tighter bounds?** `lo = ceil(sum / h)`, `hi = max(piles)` — fewer iterations.

In [32]:
def min_eating_speed(piles, h):
    """Slowest speed that finishes within h hours."""
    # speed max(piles) always finishes in len(piles) <= h hours
    lo, hi = 1, max(piles)
    while lo <= hi:
        mid = (lo + hi) // 2
        # hours needed at speed mid (ceil per pile)
        if sum((p + mid - 1) // mid for p in piles) <= h:
            hi = mid - 1
        else:
            lo = mid + 1
    return lo


assert min_eating_speed([3, 6, 7, 11], 8) == 4
assert min_eating_speed([30, 11, 23, 4, 20], 5) == 30
assert min_eating_speed([30, 11, 23, 4, 20], 6) == 23
print(min_eating_speed([3, 6, 7, 11], 8))

4


### 2.3.2 [1011. Capacity To Ship Packages Within D Days](https://leetcode.com/problems/capacity-to-ship-packages-within-d-days/) — ★★ · Medium

**Problem.** Minimum ship capacity to ship packages in order within `days` days.  
**Example.** `weights = [1,2,3,4,5,6,7,8,9,10], days = 5` → `15`  
**Spot it.** Minimum capacity such that feasible → binary search on capacity, greedy check.

**Ask first**
- Must keep the order? → Yes (otherwise it's bin packing — much harder).
- Can a package be split? → No, so capacity ≥ the heaviest package.

> 💡 **Idea.** Binary-search the capacity in `[max(w), sum(w)]`. Check: greedily fill each day in order; count days needed; OK if ≤ D.

**Brute force:** try every capacity from max to sum, O(n · sum).

**Steps**
1. Range `[max(weights), sum(weights)]` (must fit the heaviest package; one day ships all).
2. `days_needed(cap)`: load greedily, start a new day on overflow.
3. First capacity with `days_needed ≤ days`.

**Cost:** Time O(n log sum) · Space O(1)

**Likely follow-ups**
- **Split array into k parts minimising the largest sum (LC 410)?** Exactly the same code: "days" become "parts".
- **Why is the greedy check correct?** Packing as much as possible each day never uses more days than any other packing.
- **Output which packages go on which day?** Re-run the greedy with the final capacity and record the cut points.

In [33]:
def ship_within_days(weights, days):
    """Least capacity shipping everything within days."""

    def days_needed(cap):
        count, load = 1, 0
        for w in weights:
            # this package overflows today: start a new day
            if load + w > cap:
                count += 1
                load = 0
            load += w
        return count

    # must carry the heaviest package; one day carries everything
    lo, hi = max(weights), sum(weights)
    while lo <= hi:
        mid = (lo + hi) // 2
        if days_needed(mid) <= days:
            hi = mid - 1
        else:
            lo = mid + 1
    return lo


assert ship_within_days([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5) == 15
assert ship_within_days([3, 2, 2, 4, 1, 4], 3) == 6
assert ship_within_days([1, 2, 3, 1, 1], 4) == 3
print(ship_within_days([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5))

15


### 2.3.3 [378. Kth Smallest Element in a Sorted Matrix](https://leetcode.com/problems/kth-smallest-element-in-a-sorted-matrix/) — ★ · Medium

**Problem.** k-th smallest value in an `n × n` matrix whose rows and columns are sorted.  
**Example.** `matrix = [[1,5,9],[10,11,13],[12,13,15]], k = 8` → `13`  
**Spot it.** Binary search on the *value*, counting elements ≤ value with a staircase walk.

**Ask first**
- Duplicates count separately? → Yes, k-th in sorted order.

> 💡 **Idea.** Binary-search the *value*. For a guess `v`, count cells ≤ v with the staircase walk (O(n)). The smallest `v` with count ≥ k is the answer.

**Brute force:** flatten and sort, O(n² log n); or a min-heap popping k times, O(k log n).

**Steps**
1. Range `[matrix[0][0], matrix[-1][-1]]`.
2. `count_le(v)`: from the bottom-left, move right when ≤ v (adding the column count), up otherwise.
3. First `v` with `count_le(v) ≥ k`.

**Cost:** Time O(n log(max − min)) · Space O(1)

**Likely follow-ups**
- **O(1) extra memory?** The value binary search + staircase count already uses O(1).
- **Heap approach?** Push the first cell of each row; pop k times, pushing the next cell in the same row. O(k log n), O(n) space.
- **O(n) time?** Exists (Frederickson–Johnson) but is far beyond what's expected — mention it, don't code it.

In [34]:
def kth_smallest_matrix(matrix, k):
    """k-th smallest entry of a sorted matrix."""
    n = len(matrix)

    def count_le(v):
        # staircase from the bottom-left corner
        r, c, count = n - 1, 0, 0
        while r >= 0 and c < n:
            # whole column c up to row r is <= v
            if matrix[r][c] <= v:
                count += r + 1
                c += 1
            else:
                r -= 1
        return count

    lo, hi = matrix[0][0], matrix[-1][-1]
    while lo <= hi:
        mid = (lo + hi) // 2
        if count_le(mid) >= k:
            hi = mid - 1
        else:
            lo = mid + 1
    return lo


assert kth_smallest_matrix([[1, 5, 9], [10, 11, 13], [12, 13, 15]], 8) == 13
assert kth_smallest_matrix([[-5]], 1) == -5
print(kth_smallest_matrix([[1, 5, 9], [10, 11, 13], [12, 13, 15]], 8))

13


### 2.3.4 [4. Median of Two Sorted Arrays](https://leetcode.com/problems/median-of-two-sorted-arrays/) — ★★ · Hard

**Problem.** Median of two sorted arrays in O(log(min(m, n))).  
**Example.** `[1, 3]` and `[2]` → `2.0`; `[1, 2]` and `[3, 4]` → `2.5`  
**Spot it.** Binary search a *partition* of the shorter array.

**Ask first**
- Must it be O(log(m + n))? → Yes — merging (O(m + n)) is the fallback.

> 💡 **Idea.** Binary-search how many elements the shorter array contributes to the left half. The cut is right when every left element ≤ every right element.

**Brute force:** merge until the middle, O(m + n).

**Steps**
1. Make `a` the shorter array; `half = (m + n + 1) // 2`.
2. Take `i` from `a` and `j = half − i` from `b` (±∞ beyond the edges).
3. `a[i−1] > b[j]` → take fewer from `a`; `b[j−1] > a[i]` → take more.
4. When valid: odd total → max of left side; even → average of max-left and min-right.

**Cost:** Time O(log min(m, n)) · Space O(1)

**Likely follow-ups**
- **k-th smallest of two sorted arrays?** Same partition idea, or discard k/2 elements from one array each step: O(log k).
- **Median of a stream?** Two heaps (LC 295).

In [35]:
def find_median_sorted_arrays(a, b):
    """Median of the union of two sorted arrays."""
    # search the shorter array's split point
    if len(a) > len(b):
        a, b = b, a
    m, n = len(a), len(b)
    half = (m + n + 1) // 2
    lo, hi = 0, m
    inf = float("inf")
    while True:
        # i items from a and j from b form the left half
        i = (lo + hi) // 2
        j = half - i
        a_left = a[i - 1] if i > 0 else -inf
        a_right = a[i] if i < m else inf
        b_left = b[j - 1] if j > 0 else -inf
        b_right = b[j] if j < n else inf
        # too many from a
        if a_left > b_right:
            hi = i - 1
        # too few from a
        elif b_left > a_right:
            lo = i + 1
        # valid split: read the median from the boundary values
        else:
            left_max = max(a_left, b_left)
            if (m + n) % 2:
                return float(left_max)
            return (left_max + min(a_right, b_right)) / 2


assert find_median_sorted_arrays([1, 3], [2]) == 2.0
assert find_median_sorted_arrays([1, 2], [3, 4]) == 2.5
assert find_median_sorted_arrays([], [1]) == 1.0
print(find_median_sorted_arrays([1, 2], [3, 4]))

2.5


### 2.3.5 [981. Time Based Key-Value Store](https://leetcode.com/problems/time-based-key-value-store/) — ★★ · Medium

**Problem.** `set(key, value, t)`; `get(key, t)` returns the value with the largest timestamp ≤ `t`.  
**Example.** `set("love", "high", 1)`, `set("love", "low", 4)`, `get("love", 3)` → `"high"`, `get("love", 5)` → `"low"`  
**Spot it.** Timestamps arrive increasing → per-key sorted list + upper-bound search.

**Ask first**
- Do timestamps for `set` always increase? → Yes → just append. If not, you need sorted inserts.

> 💡 **Idea.** Map each key to a list of `(timestamp, value)`. Timestamps only increase, so the list is sorted — `get` binary-searches the last timestamp ≤ t.

**Brute force:** scan the key's whole list on each `get`, O(n).

**Steps**
1. `set` appends the timestamp and value to the key's two lists.
2. `get` computes `i = bisect_right(times, t) − 1`.
3. Return `values[i]`, or `""` if `i < 0`.

**Cost:** Time O(1) set, O(log n) get · Space O(total sets)

**Likely follow-ups**
- **Timestamps arrive out of order?** Keep a sorted structure per key: `bisect.insort` (O(n) insert) or a balanced tree / `SortedList` (O(log n)).
- **Support delete?** Remove from the per-key sorted list (or store a tombstone value) so later `get`s skip it.
- **Make it thread-safe?** Lock per key (or a read–write lock): many readers, one writer; appends are the only writes.
- **Memory grows forever?** Drop entries older than a retention window, or cap versions per key.

In [36]:
class TimeMap:
    """Versioned key-value store."""

    def __init__(self):
        # parallel lists per key; timestamps arrive increasing
        self.times = defaultdict(list)
        self.values = defaultdict(list)

    def set(self, key, value, timestamp):
        self.times[key].append(timestamp)
        self.values[key].append(value)

    def get(self, key, timestamp):
        # last entry with time <= timestamp
        i = bisect_right(self.times[key], timestamp) - 1
        return self.values[key][i] if i >= 0 else ""


tm = TimeMap()
tm.set("foo", "bar", 1)
assert tm.get("foo", 1) == "bar" and tm.get("foo", 3) == "bar"
tm.set("foo", "bar2", 4)
assert tm.get("foo", 4) == "bar2" and tm.get("foo", 5) == "bar2"
assert tm.get("foo", 0) == ""
print(tm.get("foo", 3))

bar


---
# 3) Linked List

**The gist.** Each node only knows the next one. You can't jump to position i — you walk and rewire `next` pointers. Every question is "rewire without losing the rest of the chain".

> **Think of it like…** a treasure hunt where each clue says where the next one is. Rewrite a clue before reading it and the rest of the hunt is gone — so always save `next` before you overwrite it.

| Tool | Use it for |
|---|---|
| **Dummy head** `dummy = ListNode(0, head)` | anything that might change or delete the head |
| **Slow/fast pointers** | middle, cycles, k-th from the end |
| **In-place reversal** `prev, cur = None, head` | reverse all / part / k-groups, reorder |

Pointer order: **save `next` → rewire → advance**. Draw three boxes on paper for anything with more than two assignments.

**Classic bugs:** rewiring `cur.next` before saving it · special-casing the head instead of using a dummy · `while fast.next` without checking `fast` first · forgetting `slow.next = None` when splitting (creates a cycle).

In [37]:
class ListNode:
    """Singly linked list node."""

    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build(values):
    """Linked list from a Python list."""
    dummy = tail = ListNode()
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def to_list(head):
    """Python list from a linked list."""
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out


print(to_list(build([1, 2, 3])))

[1, 2, 3]


## 3.1 Reversal and merging

Every reversal is the same three-pointer dance (`prev`, `cur`, `nxt`). Partial and group reversal add one more pointer: the node *before* the segment, so the reversed piece can be reconnected.

### 3.1.1 [206. Reverse Linked List](https://leetcode.com/problems/reverse-linked-list/) — ★★★ · Easy

**Problem.** Reverse a singly linked list.  
**Example.** `1→2→3→4→5` → `5→4→3→2→1`  
**Spot it.** The fundamental pointer rewiring.

**Ask first**
- Iterative or recursive? → Either; iterative is O(1) space.

> 💡 **Idea.** Walk the list flipping each `next` to point backwards: save next → flip → advance.

**Brute force:** copy values to a list and rebuild, O(n) space.

**Steps**
1. `prev = None`, `cur = head`.
2. Save `nxt = cur.next`.
3. Point `cur.next` at `prev`.
4. Advance `prev = cur`, `cur = nxt`; return `prev` at the end.

**Cost:** Time O(n) (each node rewired once) · Space O(1)

In [ ]:
def reverse_list(head):
    """Head of the reversed list."""
    prev = None
    cur = head
    while cur:
        # save the rest of the list before rewiring
        nxt = cur.next
        # point this node backwards
        cur.next = prev
        # advance both pointers
        prev, cur = cur, nxt
    return prev


assert to_list(reverse_list(build([1, 2, 3, 4, 5]))) == [5, 4, 3, 2, 1]
assert to_list(reverse_list(build([]))) == []
print(to_list(reverse_list(build([1, 2]))))

### 3.1.2 [21. Merge Two Sorted Lists](https://leetcode.com/problems/merge-two-sorted-lists/) — ★★★ · Easy

**Problem.** Merge two sorted lists into one sorted list.  
**Example.** `1→2→4` and `1→3→4` → `1→1→2→3→4→4`  
**Spot it.** Merge step of merge sort + dummy head.

**Ask first**
- Splice existing nodes or make new ones? → Splice.

> 💡 **Idea.** Dummy head; keep attaching the smaller head of the two lists; attach the leftover at the end.

**Brute force:** dump all values, sort, rebuild, O(n log n).

**Steps**
1. `dummy` and `tail` start at a placeholder node.
2. While both lists have nodes, attach the smaller head to `tail` and advance it.
3. Attach whatever list remains.
4. Return `dummy.next`.

**Cost:** Time O(m + n) · Space O(1) (nodes are reused)

In [ ]:
def merge_two_lists(a, b):
    """Merged sorted list."""
    # placeholder so the first attach needs no special case
    dummy = tail = ListNode()
    
    while a and b:
        # take the smaller head
        if a.val <= b.val:
            tail.next, a = a, a.next
        else:
            tail.next, b = b, b.next
        tail = tail.next
    
    # one list is empty; append the other as is
    tail.next = a or b
    return dummy.next


assert to_list(merge_two_lists(build([1, 2, 4]), build([1, 3, 4]))) == [
    1,
    1,
    2,
    3,
    4,
    4,
]
assert to_list(merge_two_lists(build([]), build([0]))) == [0]
print(to_list(merge_two_lists(build([1, 2, 4]), build([1, 3, 4]))))

[1, 1, 2, 3, 4, 4]


### 3.1.3 [23. Merge k Sorted Lists](https://leetcode.com/problems/merge-k-sorted-lists/) — ★★★ · Hard

**Problem.** Merge `k` sorted linked lists.  
**Example.** `[1→4→5, 1→3→4, 2→6]` → `1→1→2→3→4→4→5→6`  
**Spot it.** Repeatedly need the smallest of k heads → min-heap.

**Ask first**
- Empty lists inside? → Yes — skip `None`s when building the heap.

> 💡 **Idea.** Min-heap holding the current head of each list. Pop the smallest, attach it, push its next.

**Brute force:** merge lists one by one into the result, O(N·k).

**Steps**
1. Push `(val, index, node)` for each non-empty list head.
2. Pop the smallest and attach it to the output.
3. Push that node's successor if it exists.
4. Repeat until the heap is empty.

**Cost:** Time O(N log k) (N pops and pushes on a heap of size ≤ k) · Space O(k) (the heap)

**Likely follow-ups**
- **Without a heap?** Divide and conquer: merge pairs of lists, then pairs of results — O(N log k).
- **Lists are huge files on disk?** External k-way merge: same heap, reading a buffered chunk from each file.

In [40]:
def merge_k_lists(lists):
    """Merged sorted list of k sorted lists."""
    # index i breaks ties so nodes are never compared
    heap = [(node.val, i, node) for i, node in enumerate(lists) if node]
    heapq.heapify(heap)
    dummy = tail = ListNode()
    while heap:
        # smallest current head across all lists
        _, i, node = heapq.heappop(heap)
        tail.next = tail = node
        # its successor becomes that list's new head
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy.next


lists = [build([1, 4, 5]), build([1, 3, 4]), build([2, 6])]
assert to_list(merge_k_lists(lists)) == [1, 1, 2, 3, 4, 4, 5, 6]
assert to_list(merge_k_lists([])) == []
print("ok")

ok


### 3.1.4 [92. Reverse Linked List II](https://leetcode.com/problems/reverse-linked-list-ii/) — ★★ · Medium

**Problem.** Reverse the nodes from position `left` to `right` (1-indexed) in one pass.  
**Example.** `1→2→3→4→5, left = 2, right = 4` → `1→4→3→2→5`  
**Spot it.** Partial reversal → dummy head + "move next node to the front of the sublist".

**Ask first**
- Can `left` be 1? → Yes → use a dummy head so the head can change.

> 💡 **Idea.** Walk to the node before `left`. Then, `right − left` times, take the node after the current one and move it to the front of the segment.

**Brute force:** copy values into an array, reverse the slice, write back — O(n) space.

**Steps**
1. Walk `pre` to the node before position `left` (start at a dummy).
2. `cur = pre.next` (it will end up as the segment's tail).
3. Repeat `right − left` times: detach `cur.next` and insert it right after `pre`.
4. Return `dummy.next`.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **One pass?** The front-insertion method above is one pass, O(1) space.
- **Reverse every k nodes (LC 25)?** Repeat this per group; stop when fewer than k nodes remain.
- **Reverse alternate groups / odd-length groups (LC 2074)?** Same building block, just decide per group whether to reverse.

In [ ]:
def reverse_between(head, left, right):
    """Reverse positions left..right."""
    dummy = ListNode(0, head)
    # pre: node just before the segment
    pre = dummy
    
    for _ in range(left - 1):
        pre = pre.next
    # cur stays put and drifts to the segment's end
    cur = pre.next
    for _ in range(right - left):
        # lift the node after cur...
        moved = cur.next
        cur.next = moved.next
        # ...and insert it at the front of the segment
        moved.next = pre.next
        pre.next = moved
    return dummy.next


assert to_list(reverse_between(build([1, 2, 3, 4, 5]), 2, 4)) == [1, 4, 3, 2, 5]
assert to_list(reverse_between(build([5]), 1, 1)) == [5]
print(to_list(reverse_between(build([1, 2, 3, 4, 5]), 2, 4)))

[1, 4, 3, 2, 5]


### 3.1.5 [25. Reverse Nodes in k-Group](https://leetcode.com/problems/reverse-nodes-in-k-group/) — ★★ · Hard

**Problem.** Reverse every consecutive group of `k` nodes; a final short group stays as is.  
**Example.** `1→2→3→4→5, k = 2` → `2→1→4→3→5`  
**Spot it.** Repeated partial reversal with a "group previous" pointer.

**Ask first**
- Last group shorter than k? → Leave it as is.

> 💡 **Idea.** Check k nodes exist; reverse them; connect the previous group's tail to the new head; the old head becomes the tail for the next round.

**Brute force:** copy values into an array, reverse chunks, write back — O(n) space.

**Steps**
1. From `group_prev`, walk k nodes to find `kth`; fewer than k → stop.
2. Reverse the group, starting `prev` at `kth.next`.
3. Link `group_prev.next = kth` (new group head).
4. The old group head is now the tail: it becomes the next `group_prev`.

**Cost:** Time O(n) (each node visited twice) · Space O(1)

**Likely follow-ups**
- **Reverse the leftover partial group too?** Drop the "k nodes exist" check.

In [ ]:
def reverse_k_group(head, k):
    """Reverse nodes in groups of k."""
    dummy = ListNode(0, head)
    group_prev = dummy
    
    while True:
        # find the k-th node of this group, or stop if the group is short
        kth = group_prev
        for _ in range(k):
            kth = kth.next
            if not kth:
                return dummy.next
        group_next = kth.next
        # reverse the group; its tail ends up pointing at group_next
        prev, cur = group_next, group_prev.next
        while cur is not group_next:
            nxt = cur.next
            cur.next = prev
            prev, cur = cur, nxt
        # reconnect the front and move to the next group
        old_head = group_prev.next
        group_prev.next = kth
        group_prev = old_head


assert to_list(reverse_k_group(build([1, 2, 3, 4, 5]), 2)) == [2, 1, 4, 3, 5]
assert to_list(reverse_k_group(build([1, 2, 3, 4, 5]), 3)) == [3, 2, 1, 4, 5]
print(to_list(reverse_k_group(build([1, 2, 3, 4, 5, 6]), 3)))

[3, 2, 1, 6, 5, 4]


### 3.1.6 [2. Add Two Numbers](https://leetcode.com/problems/add-two-numbers/) — ★★ · Medium

**Problem.** Digits stored in reverse order; return the sum as a list.  
**Example.** `2→4→3` + `5→6→4` → `7→0→8` (342 + 465 = 807)  
**Spot it.** Grade-school addition from the least significant digit, carry propagates.

**Ask first**
- Different lengths? → Yes — treat a missing digit as 0, and keep the last carry (`5 + 5 = 10`).

> 💡 **Idea.** Digits are stored ones-first, so add like on paper: digit + digit + carry, write `sum % 10`, carry `sum // 10`. Don't forget a final carry.

**Brute force:** convert both lists to integers, add, convert back — overflows in fixed-width languages.

**Steps**
1. Loop while either list has nodes or `carry` is non-zero.
2. `total = carry + a.val + b.val` (missing digits count as 0).
3. `carry, digit = divmod(total, 10)`; append `digit`.
4. Advance whichever lists remain.

**Cost:** Time O(max(m, n)) · Space O(max(m, n)) (the result)

**Likely follow-ups**
- **Digits stored most-significant first (LC 445)?** Without reversing: push both lists onto stacks, pop to add from the ones place, and build the result by inserting at the front.
- **Subtract or multiply two lists?** Same digit-by-digit idea with borrow (subtract) or the grade-school multiply into a result array.
- **Do it recursively?** Recurse to the end (after padding the shorter list with zeros), return the carry upwards.

In [43]:
def add_two_numbers(a, b):
    """Sum of two reversed-digit numbers."""
    dummy = tail = ListNode()
    carry = 0
    # keep going while digits or a carry remain
    while a or b or carry:
        total = carry + (a.val if a else 0) + (b.val if b else 0)
        carry, digit = divmod(total, 10)
        tail.next = tail = ListNode(digit)
        a = a.next if a else None
        b = b.next if b else None
    return dummy.next


assert to_list(add_two_numbers(build([2, 4, 3]), build([5, 6, 4]))) == [7, 0, 8]
assert to_list(add_two_numbers(build([9] * 7), build([9] * 4))) == [
    8,
    9,
    9,
    9,
    0,
    0,
    0,
    1,
]
print(to_list(add_two_numbers(build([2, 4, 3]), build([5, 6, 4]))))

[7, 0, 8]


### 3.1.7 [148. Sort List](https://leetcode.com/problems/sort-list/) — ★★ · Medium

**Problem.** Sort a linked list in O(n log n).  
**Example.** `4→2→1→3` → `1→2→3→4`  
**Spot it.** Linked lists favour merge sort (no random access needed).

**Ask first**
- O(1) extra space required? → The follow-up says yes → bottom-up merge sort (no recursion stack).

> 💡 **Idea.** Merge sort: find the middle with slow/fast, cut, sort each half, merge two sorted lists.

**Brute force:** copy values to an array, sort, rebuild — O(n) space.

**Steps**
1. Base case: 0 or 1 node.
2. Find the middle with slow/fast and cut the list in two.
3. Sort each half recursively.
4. Merge the two sorted halves.

**Cost:** Time O(n log n) (log n levels, O(n) merging per level) · Space O(log n) (recursion depth (bottom-up merging achieves O(1)))

**Likely follow-ups**
- **O(1) space?** Bottom-up: merge runs of size 1, then 2, 4, 8… walking the list each pass, cutting and splicing with a dummy head. O(n log n) time, O(1) space.
- **Why not quicksort on a list?** No random access for good pivots, and it's unstable; merge sort fits lists naturally.
- **List is almost sorted?** Natural merge sort: merge existing ascending runs instead of fixed sizes.

In [44]:
def sort_list(head):
    """Merge-sorted list."""
    if not head or not head.next:
        return head
    # slow stops at the end of the first half
    slow, fast = head, head.next
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    # cut the list into two halves
    right = slow.next
    slow.next = None
    return merge_two_lists(sort_list(head), sort_list(right))


assert to_list(sort_list(build([4, 2, 1, 3]))) == [1, 2, 3, 4]
assert to_list(sort_list(build([-1, 5, 3, 4, 0]))) == [-1, 0, 3, 4, 5]
print(to_list(sort_list(build([4, 2, 1, 3]))))

[1, 2, 3, 4]


## 3.2 Slow and fast pointers

Different speeds (1 and 2 steps) find middles and cycles; a fixed gap (n steps) finds the n-th node from the end.

### 3.2.1 [876. Middle of the Linked List](https://leetcode.com/problems/middle-of-the-linked-list/) — ★★ · Easy

**Problem.** Return the middle node (second middle if even length).  
**Example.** `1→2→3→4→5` → the node `3`; `1→2→3→4→5→6` → the node `4`  
**Spot it.** Fast moves two steps per slow step → slow ends in the middle.

**Ask first**
- Even length → which middle? → The second one (loop while `fast and fast.next`).

> 💡 **Idea.** Fast moves 2, slow moves 1. When fast falls off, slow is in the middle.

**Brute force:** count the length, walk half — two passes.

**Steps**
1. Start both at the head.
2. Move slow one step and fast two steps while fast and fast.next exist.
3. Return slow.

**Cost:** Time O(n) · Space O(1)

In [45]:
def middle_node(head):
    """Middle node (second of two middles)."""
    slow = fast = head
    # fast covers twice the distance of slow
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    return slow


assert to_list(middle_node(build([1, 2, 3, 4, 5]))) == [3, 4, 5]
assert to_list(middle_node(build([1, 2, 3, 4, 5, 6]))) == [4, 5, 6]
print(middle_node(build([1, 2, 3])).val)

2


### 3.2.2 [141. Linked List Cycle](https://leetcode.com/problems/linked-list-cycle/) — ★★★ · Easy

**Problem.** Does the list contain a cycle? O(1) memory.  
**Example.** `3→2→0→-4` where the last node links back to `2` → `True`  
**Spot it.** Floyd's tortoise and hare.

**Ask first**
- O(1) memory? → Yes → Floyd, not a visited set.

> 💡 **Idea.** Fast (2 steps) and slow (1 step). If there's a cycle, fast laps slow and they meet.

**Brute force:** a set of visited nodes, O(n) space.

**Steps**
1. Slow moves 1, fast moves 2.
2. If they ever point to the same node, there is a cycle.
3. If fast reaches `None`, there is none.

**Cost:** Time O(n) · Space O(1)

In [46]:
def has_cycle(head):
    """True if the list loops."""
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        # fast lapped slow: there is a loop
        if slow is fast:
            return True
    # fast fell off the end
    return False


def build_cycle(values, pos):
    """List whose tail links back to index pos (-1 for none)."""
    head = build(values)
    if pos >= 0:
        nodes = []
        node = head
        while node:
            nodes.append(node)
            node = node.next
        # close the loop
        nodes[-1].next = nodes[pos]
    return head


assert has_cycle(build_cycle([3, 2, 0, -4], 1))
assert not has_cycle(build_cycle([1], -1))
print(has_cycle(build_cycle([1, 2], 0)))

True


### 3.2.3 [142. Linked List Cycle II](https://leetcode.com/problems/linked-list-cycle-ii/) — ★★ · Medium

**Problem.** Return the node where the cycle begins, or `None`.  
**Example.** the same list (`-4` links back to `2`) → the node with value `2`  
**Spot it.** Floyd phase 2: after meeting, a pointer from the head and one from the meeting point meet at the cycle start.

**Ask first**
- Can I modify the list (e.g. mark nodes)? → No, and O(1) memory → Floyd.

> 💡 **Idea.** After slow and fast meet, move one pointer back to the head. Step both one at a time — they meet exactly at the cycle's entrance.

**Brute force:** a visited set — the first repeated node is the entrance. O(n) space.

**Steps**
1. Run slow/fast until they meet (or fast ends → no cycle).
2. Start `finder` at the head.
3. Move `finder` and `slow` one step at a time.
4. They meet at the cycle start.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **Why does resetting to the head work?** If the entry is `a` steps from the head and they met `b` steps into the cycle of length `c`, then `a ≡ c − b (mod c)`: walking `a` steps from the head or from the meeting point both land on the entry.
- **Length of the cycle?** From the meeting point, walk until you're back, counting steps.
- **Two lists — where do they intersect (LC 160)?** Walk A then B and B then A; the pointers meet at the junction (or both hit `None`).

In [47]:
def detect_cycle(head):
    """First node of the cycle, if any."""
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            # head and meeting point are the same distance from the cycle start
            finder = head
            while finder is not slow:
                finder, slow = finder.next, slow.next
            return finder
    return None


assert detect_cycle(build_cycle([3, 2, 0, -4], 1)).val == 2
assert detect_cycle(build_cycle([1], -1)) is None
print(detect_cycle(build_cycle([1, 2], 0)).val)

1


### 3.2.4 [287. Find the Duplicate Number](https://leetcode.com/problems/find-the-duplicate-number/) — ★★ · Medium

**Problem.** `n + 1` integers in `1..n`, one value repeated; find it with O(1) space, no modification.  
**Example.** `[1, 3, 4, 2, 2]` → `2`  
**Spot it.** Treat `i → nums[i]` as a linked list; the duplicate is the cycle entrance.

**Ask first**
- Can I modify the array? → No (rules out sorting and sign-marking).
- O(1) extra space? → Yes (rules out a set).

> 💡 **Idea.** Treat the array as a linked list: index `i` points to `nums[i]`. The duplicate value is where two indices point to the same place — the start of a cycle. Use Floyd.

**Brute force:** a set, O(n) space; or sort, O(n log n) and modifies the input.

**Steps**
1. Slow `x = nums[x]`, fast `x = nums[nums[x]]`, starting from index 0, until they meet.
2. Start `finder` at 0.
3. Advance `finder` and `slow` one step at a time until they meet.
4. That value is the duplicate.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **Prove a duplicate must exist.** Pigeonhole: n + 1 numbers, only n possible values.
- **Another O(1)-space way?** Binary search on the value: count how many numbers are ≤ mid; if more than mid, the duplicate is ≤ mid. O(n log n).
- **Modification allowed?** Sign-marking (`nums[abs(x)] *= -1`) or cyclic sort, O(n) time, O(1) space.

In [48]:
def find_duplicate(nums):
    """The repeated value."""
    # phase 1: meet inside the cycle of i -> nums[i]
    slow = fast = 0
    while True:
        slow, fast = nums[slow], nums[nums[fast]]
        if slow == fast:
            break
    # phase 2: walk to the cycle entrance (the duplicate)
    finder = 0
    while finder != slow:
        finder, slow = nums[finder], nums[slow]
    return finder


assert find_duplicate([1, 3, 4, 2, 2]) == 2
assert find_duplicate([3, 1, 3, 4, 2]) == 3
assert find_duplicate([3, 3, 3, 3, 3]) == 3
print(find_duplicate([1, 3, 4, 2, 2]))

2


### 3.2.5 [19. Remove Nth Node From End of List](https://leetcode.com/problems/remove-nth-node-from-end-of-list/) — ★★★ · Medium

**Problem.** Remove the n-th node from the end in one pass.  
**Example.** `1→2→3→4→5, n = 2` → `1→2→3→5`  
**Spot it.** Gap of n between two pointers.

**Ask first**
- Can the head be the one removed? → Yes → dummy head.

> 💡 **Idea.** Move a lead pointer n steps ahead, then move both until the lead hits the end — the trailing pointer sits just before the node to delete.

**Brute force:** count the length, then walk to `len − n` — two passes.

**Steps**
1. Start both at a dummy before the head.
2. Move `fast` n steps ahead.
3. Move both until `fast.next` is `None`.
4. Unlink `slow.next`.

**Cost:** Time O(L) · Space O(1)

**Likely follow-ups**
- **One pass?** That's the gap method above.
- **n might be larger than the length?** Check while advancing the lead; if it runs out, return the list unchanged (or raise — ask).
- **Doubly linked list?** Walk from the tail if you have it; otherwise same gap trick.

In [49]:
def remove_nth_from_end(head, n):
    """List without its n-th node from the end."""
    # dummy makes deleting the head a normal case
    dummy = ListNode(0, head)
    slow = fast = dummy
    # open a gap of n nodes
    for _ in range(n):
        fast = fast.next
    # slide until fast is the last node
    while fast.next:
        slow, fast = slow.next, fast.next
    # slow is just before the target
    slow.next = slow.next.next
    return dummy.next


assert to_list(remove_nth_from_end(build([1, 2, 3, 4, 5]), 2)) == [1, 2, 3, 5]
assert to_list(remove_nth_from_end(build([1]), 1)) == []
assert to_list(remove_nth_from_end(build([1, 2]), 2)) == [2]
print(to_list(remove_nth_from_end(build([1, 2, 3, 4, 5]), 2)))

[1, 2, 3, 5]


### 3.2.6 [143. Reorder List](https://leetcode.com/problems/reorder-list/) — ★★★ · Medium

**Problem.** Reorder `L0 → L1 → … → Ln` into `L0 → Ln → L1 → Ln−1 → …` in place.  
**Example.** `1→2→3→4` → `1→4→2→3`  
**Spot it.** Middle + reverse second half + interleave merge.

**Ask first**
- Must I rewire nodes (not swap values)? → Yes, in place.

> 💡 **Idea.** Three steps you already know: find the middle, reverse the second half, then weave the two halves together.

**Brute force:** put all nodes in an array and link `0, n−1, 1, n−2, …`, O(n) space.

**Steps**
1. Find the middle; cut the list after it.
2. Reverse the second half.
3. Alternately take one node from each half until the second is used up.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **O(1) space?** The three-step method is O(1).
- **Recursive version?** Possible (recurse to the tail, link on the way back) but uses O(n) stack — mention the trade-off.
- **Check if a list is a palindrome (LC 234)?** Same first two steps; compare the halves; optionally reverse back to restore.

In [50]:
def reorder_list(head):
    """Interleave front and reversed back halves in place."""
    # slow ends at the last node of the first half
    slow, fast = head, head.next
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    # detach and reverse the second half
    second = reverse_list(slow.next)
    slow.next = None
    # zip: one node from each half in turn
    first = head
    while second:
        f_next, s_next = first.next, second.next
        first.next, second.next = second, f_next
        first, second = f_next, s_next


h = build([1, 2, 3, 4, 5])
reorder_list(h)
assert to_list(h) == [1, 5, 2, 4, 3]
h = build([1, 2, 3, 4])
reorder_list(h)
assert to_list(h) == [1, 4, 2, 3]
print(to_list(h))

[1, 4, 2, 3]


## 3.3 Rewiring and deletion

Deleting or moving nodes is either "skip over" (`prev.next = prev.next.next`) or "build new chains off dummy heads and join them". End the last chain with `None`.

### 3.3.1 [138. Copy List with Random Pointer](https://leetcode.com/problems/copy-list-with-random-pointer/) — ★★ · Medium

**Problem.** Deep-copy a list where each node also has a `random` pointer.  
**Example.** `[[7, null], [13, 0]]` (each pair is `[value, index of its random node]`) → a copy with the same values and pointers, but brand-new nodes  
**Spot it.** Map old node → new node, then wire pointers.

**Ask first**
- O(1) extra space? → Possible with interleaving (below); the map version is O(n) and easier.

> 💡 **Idea.** Map each original node to its copy. First pass: create all copies. Second pass: wire `next` and `random` through the map.

**Brute force:** for each random pointer, find its index by walking from the head, O(n²).

**Steps**
1. First pass: map every original node to a new node (`None → None` too).
2. Second pass: `copy.next = map[old.next]`, `copy.random = map[old.random]`.
3. Return `map[head]`.

**Cost:** Time O(n) · Space O(n) (the map (an interleaving trick gets O(1)))

**Likely follow-ups**
- **O(1) extra space?** Interleave: insert each copy right after its original (`A → A' → B → B'`), set `copy.random = orig.random.next`, then unzip the two lists.
- **Clone a general graph (LC 133)?** Same map idea with DFS/BFS: map old → new, create on first visit.
- **Recursive one-pass?** `copy(node)`: if in map, return it; else create, store, then set `next = copy(node.next)`, `random = copy(node.random)` — O(n) stack.

In [51]:
class RandomNode:
    """Node with an extra random pointer."""

    def __init__(self, val):
        self.val = val
        self.next = None
        self.random = None


def copy_random_list(head):
    """Deep copy preserving random links."""
    # old node -> its copy; None maps to None so links can be copied blindly
    copies = {None: None}
    node = head
    while node:
        copies[node] = RandomNode(node.val)
        node = node.next
    # every copy exists now: wire both pointers through the map
    node = head
    while node:
        copies[node].next = copies[node.next]
        copies[node].random = copies[node.random]
        node = node.next
    return copies[head]


nodes = [RandomNode(v) for v in [7, 13, 11, 10, 1]]
for a, b in zip(nodes, nodes[1:]):
    a.next = b
for node, r in zip(nodes, [None, 0, 4, 2, 0]):
    node.random = nodes[r] if r is not None else None
copy = copy_random_list(nodes[0])
out = []
while copy:
    out.append((copy.val, copy.random.val if copy.random else None))
    assert copy not in nodes
    copy = copy.next
assert out == [(7, None), (13, 7), (11, 1), (10, 11), (1, 7)]
print(out)

[(7, None), (13, 7), (11, 1), (10, 11), (1, 7)]


## 3.4 Linked-list design

Hash map (O(1) *find*) + doubly linked list (O(1) *move / remove* once you have the node). Sentinel head and tail nodes kill every edge case.

### 3.4.1 [146. LRU Cache](https://leetcode.com/problems/lru-cache/) — ★★ · Medium

**Problem.** Fixed-capacity cache with O(1) `get` and `put`, evicting the least recently used key.  
**Example.** capacity 2: `put(1, 1)`, `put(2, 2)`, `get(1)` → `1`, `put(3, 3)` (throws out key 2), `get(2)` → `-1`  
**Spot it.** O(1) lookup + O(1) recency order → hash map + doubly linked list.

**Ask first**
- Does `put` on an existing key count as a use? → Yes → update and move to front.
- Thread-safe? → Not here — but expect it as a follow-up.

> 💡 **Idea.** Hash map key → node, plus a doubly linked list in recency order. `get`: move the node to the front. `put`: insert/move to the front; if over capacity, cut the node at the back.

**Brute force:** a list ordered by recency — O(n) to move/remove.

**Steps**
1. Map `key → node`; list ordered least → most recent between `head` and `tail` sentinels.
2. `get`: if present, move the node to the end and return its value.
3. `put`: remove an existing node, append the new node at the end.
4. Over capacity → remove the node after `head` and delete its key.

**Cost:** Time O(1) per operation · Space O(capacity)

**Likely follow-ups**
- **Make it thread-safe?** Both `get` and `put` change the order, so one lock around each (a read–write lock doesn't help). For scale: shard into N caches by key hash, each with its own lock.
- **Add a TTL per entry?** Store `expires_at` in the node; treat expired entries as missing on `get` (lazy), and/or sweep with a min-heap by expiry.
- **Evict least *frequently* used (LFU, LC 460)?** Map key → node, map freq → doubly linked list, and track `min_freq`. All O(1).
- **In Python without writing a list?** `OrderedDict` with `move_to_end` and `popitem(last=False)`.
- **Distributed cache?** Consistent hashing across nodes, each running LRU; that becomes a system-design talk.

In [52]:
class DNode:
    """Doubly linked node for the cache."""

    def __init__(self, key=0, val=0):
        self.key, self.val = key, val
        self.prev = None
        self.next = None


class LRUCache:
    """Least-recently-used cache."""

    def __init__(self, capacity):
        self.capacity = capacity
        self.map = {}
        # sentinels: head.next = least recent, tail.prev = most recent
        self.head, self.tail = DNode(), DNode()
        self.head.next, self.tail.prev = self.tail, self.head

    def _remove(self, node):
        # neighbours skip over node
        node.prev.next, node.next.prev = node.next, node.prev

    def _append(self, node):
        # insert just before the tail sentinel
        node.prev, node.next = self.tail.prev, self.tail
        self.tail.prev.next = node
        self.tail.prev = node

    def get(self, key):
        if key not in self.map:
            return -1
        # mark as most recently used
        node = self.map[key]
        self._remove(node)
        self._append(node)
        return node.val

    def put(self, key, value):
        if key in self.map:
            self._remove(self.map[key])
        node = DNode(key, value)
        self.map[key] = node
        self._append(node)
        # evict the least recently used entry
        if len(self.map) > self.capacity:
            lru = self.head.next
            self._remove(lru)
            del self.map[lru.key]


cache = LRUCache(2)
cache.put(1, 1)
cache.put(2, 2)
assert cache.get(1) == 1
cache.put(3, 3)
assert cache.get(2) == -1
cache.put(4, 4)
assert [cache.get(1), cache.get(3), cache.get(4)] == [-1, 3, 4]
print("ok")

ok


---
# Summary

| Pattern | Recognise it by | Core move | Cost |
|---|---|---|---|
| Matching stack | brackets, paths, nested decode, expression | push opens/context, pop on close | O(n) |
| Monotonic stack | next greater/smaller, span, histogram | pop while new element breaks the order | O(n) |
| Contribution counting | sum over all subarrays of min/max | previous/next smaller distances | O(n) |
| Lower-bound binary search | sorted lookup, insert position, first/last | `while lo < hi`, `hi = mid` on true | O(log n) |
| Rotated search | rotated sorted array | one half is always sorted | O(log n) |
| Search on answer | "minimum capacity / speed / day such that…" | binary search the value, greedy feasibility | O(n log R) |
| Partition search | median of two arrays | binary search the cut of the shorter array | O(log min) |
| Dummy head | head may change | `dummy = ListNode(0, head)` | — |
| Slow/fast | middle, cycle, k-th from end, duplicate number | 2× speed or fixed gap | O(n), O(1) |
| Reversal | reverse all/part/groups, palindrome, reorder | save next → rewire → advance | O(n), O(1) |
| Map + doubly linked list | LRU cache | O(1) lookup + O(1) reorder | O(1) |